# Neuro-Symbolic Real Estate Advisory - Chuẩn hoá dữ liệu & xây tập luật Symbolic v2

Notebook này thực hiện quy trình:

```text
3 file Excel crawl BĐS
→ Gộp dữ liệu, bỏ nguồn/source khỏi output
→ Chuẩn hoá giá, diện tích, vị trí, pháp lý, tiện ích, cấu trúc nhà
→ Xây symbolic features chặt hơn
→ Xây rule base có trọng số, bằng chứng và cảnh báo rủi ro
→ Sinh triples cho Knowledge Graph
→ Test tư vấn và so sánh BĐS
```

Điểm khác bản trước:

- Không xuất các cột `source`, `source_name`, `source_file`, `source_sheet`.
- Rule symbolic chặt hơn: có `risk_flags`, `data_quality_score`, `rule_strength`, `evidence`.
- Có phân biệt `hard constraints` và `soft constraints` khi khuyến nghị.
- Giữ `URL` để đối chiếu tin đăng. Nếu muốn bỏ URL, xoá `url` khỏi `EXPORT_COLUMNS` ở cell lưu file.


#Data

In [1]:
# Cell 1 - Cài thư viện và import
!pip -q install openpyxl

import os
import re
import json
import math
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)


In [2]:
# Cell 2 - Cấu hình đường dẫn dữ liệu
# Cách dùng trên Colab:
# 1) Upload 3 file Excel lên /content
# 2) Hoặc mount Google Drive và đặt file trong MyDrive

# from google.colab import drive
# drive.mount('/content/drive')

INPUT_FILES = [
    "Crawl_data_batdongsan.xlsx",
    "Crawl_data_nhatot.xlsx",
    "Crawl_data_refine.xlsx",
]

# Notebook sẽ tự tìm file trong các thư mục này
SEARCH_ROOTS = [
    Path("/content"),
    Path("/content/drive/MyDrive"),
]

# Các sheet hợp lệ trong file crawl
VALID_SHEETS = [
    "refined_apartments",
    "Batdongsan.com.vn",
    "nhatot.com",
]

OUTPUT_DIR = Path("/content/bds_symbolic_output_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)


OUTPUT_DIR: /content/bds_symbolic_output_v2


In [3]:
# Cell 3 - Load dữ liệu Excel, nhưng KHÔNG đưa source vào output

def find_file(filename: str) -> Path | None:
    for root in SEARCH_ROOTS:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0]
    return None


def read_all_excel_files(input_files: List[str], valid_sheets: List[str]) -> pd.DataFrame:
    frames = []
    loaded_log = []

    for fname in input_files:
        path = find_file(fname)
        if path is None:
            print(f"Không tìm thấy file: {fname}")
            continue

        xls = pd.ExcelFile(path)
        for sheet in valid_sheets:
            if sheet not in xls.sheet_names:
                continue

            df = pd.read_excel(path, sheet_name=sheet)
            if len(df) == 0:
                continue

            # Không thêm source/source_name/source_file vào DataFrame output.
            frames.append(df)
            loaded_log.append({"file": fname, "sheet": sheet, "rows": len(df)})

    if len(frames) == 0:
        raise FileNotFoundError("Không load được dữ liệu. Hãy kiểm tra tên file hoặc đường dẫn upload.")

    raw = pd.concat(frames, ignore_index=True)
    print("Log load dữ liệu, chỉ để kiểm tra, không lưu vào output:")
    display(pd.DataFrame(loaded_log))
    print("Tổng số dòng raw trước chuẩn hóa:", len(raw))
    return raw

raw_df = read_all_excel_files(INPUT_FILES, VALID_SHEETS)
print("Các cột có trong dữ liệu:")
print(list(raw_df.columns))
display(raw_df.head(3))


Log load dữ liệu, chỉ để kiểm tra, không lưu vào output:


,file,sheet,rows
0,Crawl_data_batdongsan.xlsx,refined_apartments,132
1,Crawl_data_batdongsan.xlsx,Batdongsan.com.vn,200
2,Crawl_data_batdongsan.xlsx,nhatot.com,237
3,Crawl_data_nhatot.xlsx,refined_apartments,132
4,Crawl_data_nhatot.xlsx,Batdongsan.com.vn,200
5,Crawl_data_nhatot.xlsx,nhatot.com,237
6,Crawl_data_refine.xlsx,refined_apartments,132
7,Crawl_data_refine.xlsx,Batdongsan.com.vn,200
8,Crawl_data_refine.xlsx,nhatot.com,237


Tổng số dòng raw trước chuẩn hóa: 1707
Các cột có trong dữ liệu:
['Vi_Tri', 'Dien_Tich', 'Gia', 'Tien_Ich', 'Tieu_De', 'So_PN', 'So_PT', 'So_Tang', 'Huong_Nha', 'Huong_BC', 'Mat_Tien', 'Duong_Vao', 'Phap_Ly', 'Noi_That', 'Mo_Ta', 'URL']


,Vi_Tri,Dien_Tich,Gia,Tien_Ich,Tieu_De,So_PN,So_PT,So_Tang,Huong_Nha,Huong_BC,Mat_Tien,Duong_Vao,Phap_Ly,Noi_That,Mo_Ta,URL
0,Phan Huy Ích,Ngang 3.5m dài 15m,5.95,Diện Tích : Ngang 3.5m dài 15m\r\n; Hướng : Tây Nam\r\n; Vị Trí : Phan Huy Ích - P.12 - Gò Vâp - HCM\r\n; Đường : 3m...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ĐƯỜNG Bùi Quang Là,Chưa xác định,5.35,Căn hộ tiêu chuẩn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,thông 6m khu bàn cờ.\r\n,16x29,28 tỷ,Căn hộ tiêu chuẩn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Cell 4 - Hàm xử lý text, số, giá, diện tích

def clean_text(x: Any) -> str:
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()


def remove_accents(text: Any) -> str:
    text = str(text)
    text = text.replace("đ", "d").replace("Đ", "D")
    text = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")


def norm_text(text: Any) -> str:
    return remove_accents(clean_text(text)).lower()


def vn_float(x: Any) -> float:
    if x is None or pd.isna(x):
        return np.nan
    s = str(x).strip().replace(" ", "")
    # Nếu có cả dấu . và , thì giả định . là phân tách nghìn, , là thập phân
    if "." in s and "," in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(",", ".")
    s = re.sub(r"[^0-9.\-]", "", s)
    if s in ["", ".", "-"]:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def first_number(text: Any) -> float:
    s = clean_text(text)
    m = re.search(r"(\d+(?:[\.,]\d+)?)", s)
    if not m:
        return np.nan
    return vn_float(m.group(1))


def all_numbers(text: Any) -> List[float]:
    s = clean_text(text)
    nums = []
    for m in re.finditer(r"(\d+(?:[\.,]\d+)?)", s):
        val = vn_float(m.group(1))
        if not pd.isna(val):
            nums.append(val)
    return nums


def parse_area_m2(raw_area: Any, text_all: str = "") -> float:
    """
    Chuẩn hóa diện tích về m2.
    Ưu tiên cột Dien_Tich, sau đó tìm trong mô tả.
    """
    if isinstance(raw_area, (int, float)) and not pd.isna(raw_area):
        val = float(raw_area)
        return val if 10 <= val <= 2000 else np.nan

    candidates = [raw_area, text_all]

    for text in candidates:
        s = norm_text(text).replace("㎡", "m2").replace("m²", "m2")

        # 76 m2, DT: 76m2, công nhận 95m2
        m = re.search(r"(?:dt|dien tich|cong nhan|cn)?\s*[:\-]?\s*(\d+(?:[\.,]\d+)?)\s*m2", s)
        if m:
            val = vn_float(m.group(1))
            if 10 <= val <= 2000:
                return val

        # 5x20 hoặc 5,7 x 13,3
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(?:m)?\s*[x×]\s*(\d+(?:[\.,]\d+)?)\s*(?:m)?", s)
        if m:
            a, b = vn_float(m.group(1)), vn_float(m.group(2))
            if not pd.isna(a) and not pd.isna(b):
                area = a * b
                if 10 <= area <= 2000:
                    return area

        # ngang 3.5m dài 15m / rộng 4m dài 20m
        m = re.search(r"(?:ngang|rong)\s*(\d+(?:[\.,]\d+)?)\s*m?.{0,30}(?:dai|sau)\s*(\d+(?:[\.,]\d+)?)\s*m?", s)
        if m:
            a, b = vn_float(m.group(1)), vn_float(m.group(2))
            if not pd.isna(a) and not pd.isna(b):
                area = a * b
                if 10 <= area <= 2000:
                    return area

    return np.nan


def parse_price_billion(raw_price: Any, text_all: str = "", area_m2: float | None = None) -> float:
    """
    Chuẩn hóa giá về tỷ VNĐ.
    Hỗ trợ: 8,9 tỷ; 900 triệu; 120 triệu/m2; số dạng 5.95 trong refined data.
    """
    if isinstance(raw_price, (int, float)) and not pd.isna(raw_price):
        val = float(raw_price)
        return val if 0.05 <= val <= 500 else np.nan

    candidates = [raw_price, text_all]

    for text in candidates:
        s_raw = clean_text(text)
        s = norm_text(s_raw)
        if s == "" or any(k in s for k in ["thoa thuan", "lien he", "lien hệ"]):
            continue

        # 120 triệu/m2
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*trieu\s*/\s*m2", s)
        if m and area_m2 is not None and not pd.isna(area_m2):
            val = vn_float(m.group(1))
            price = val * area_m2 / 1000
            if 0.05 <= price <= 500:
                return price

        # 8,9 tỷ / 8.9 tỉ
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(ty|tỷ|ti|tỉ)", s)
        if m:
            val = vn_float(m.group(1))
            if 0.05 <= val <= 500:
                return val

        # 900 triệu
        m = re.search(r"(\d+(?:[\.,]\d+)?)\s*trieu", s)
        if m:
            val = vn_float(m.group(1)) / 1000
            if 0.05 <= val <= 500:
                return val

    # Nếu raw_price chỉ là số dạng text, mặc định là tỷ
    val = first_number(raw_price)
    if not pd.isna(val) and 0.05 <= val <= 500:
        return val

    return np.nan


def parse_int_value(raw: Any, text_all: str = "", keywords: List[str] | None = None) -> float:
    """Parse số nguyên từ cột hoặc từ text theo keywords."""
    val = first_number(raw)
    if not pd.isna(val):
        return int(round(val))

    if keywords:
        s = norm_text(text_all)
        for kw in keywords:
            # Ví dụ: số phòng ngủ: 3 phòng, 3PN
            pattern = rf"(?:{kw})\s*[:\-]?\s*(\d+)"
            m = re.search(pattern, s)
            if m:
                return int(m.group(1))

        # Case 2PN, 3PN
        if any("phong ngu" in kw or "pn" in kw for kw in keywords):
            m = re.search(r"(\d+)\s*pn", s)
            if m:
                return int(m.group(1))

    return np.nan


def parse_meter(raw: Any, text_all: str = "", mode: str = "generic") -> float:
    val = first_number(raw)
    if not pd.isna(val) and 0 < val <= 100:
        return val

    s = norm_text(text_all)

    if mode == "frontage":
        patterns = [
            r"mat tien\s*[:\-]?\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"ngang\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"mt\s*(\d+(?:[\.,]\d+)?)\s*m",
        ]
    elif mode == "road":
        patterns = [
            r"duong\s*[:\-]?\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"hem\s*(?:xe hoi|oto)?\s*(\d+(?:[\.,]\d+)?)\s*m",
            r"duong rong\s*(\d+(?:[\.,]\d+)?)\s*m",
        ]
    else:
        patterns = [r"(\d+(?:[\.,]\d+)?)\s*m"]

    for pat in patterns:
        m = re.search(pat, s)
        if m:
            val = vn_float(m.group(1))
            if not pd.isna(val) and 0 < val <= 100:
                return val

    return np.nan


In [5]:
# Cell 5 - Chuẩn hóa vị trí, loại BĐS, pháp lý, tiện ích

DISTRICT_CANONICAL = {
    "quan 1": "Quận 1", "q1": "Quận 1", "q.1": "Quận 1",
    "quan 2": "Quận 2", "q2": "Quận 2", "q.2": "Quận 2",
    "quan 3": "Quận 3", "q3": "Quận 3", "q.3": "Quận 3",
    "quan 4": "Quận 4", "q4": "Quận 4", "q.4": "Quận 4",
    "quan 5": "Quận 5", "q5": "Quận 5", "q.5": "Quận 5",
    "quan 6": "Quận 6", "q6": "Quận 6", "q.6": "Quận 6",
    "quan 7": "Quận 7", "q7": "Quận 7", "q.7": "Quận 7",
    "quan 8": "Quận 8", "q8": "Quận 8", "q.8": "Quận 8",
    "quan 9": "Quận 9", "q9": "Quận 9", "q.9": "Quận 9",
    "quan 10": "Quận 10", "q10": "Quận 10", "q.10": "Quận 10",
    "quan 11": "Quận 11", "q11": "Quận 11", "q.11": "Quận 11",
    "quan 12": "Quận 12", "q12": "Quận 12", "q.12": "Quận 12",
    "binh thanh": "Bình Thạnh",
    "phu nhuan": "Phú Nhuận",
    "go vap": "Gò Vấp",
    "go vap ": "Gò Vấp",
    "tan binh": "Tân Bình",
    "tan phu": "Tân Phú",
    "binh tan": "Bình Tân",
    "thu duc": "Thủ Đức",
    "tp thu duc": "Thủ Đức",
    "thanh pho thu duc": "Thủ Đức",
    "nha be": "Nhà Bè",
    "binh chanh": "Bình Chánh",
    "hoc mon": "Hóc Môn",
    "cu chi": "Củ Chi",
    "can gio": "Cần Giờ",
}

CENTRAL_DISTRICTS = {"Quận 1", "Quận 3", "Quận 5", "Quận 10", "Bình Thạnh", "Phú Nhuận"}
DEVELOPING_DISTRICTS = {"Thủ Đức", "Quận 2", "Quận 7", "Quận 9", "Nhà Bè", "Bình Chánh"}
SUBURBAN_DISTRICTS = {"Hóc Môn", "Củ Chi", "Cần Giờ", "Bình Tân", "Quận 12"}


def extract_district(location_text: Any) -> str:
    s = norm_text(location_text)

    # Quận số
    m = re.search(r"\b(?:quan|q\.?|q)\s*(\d{1,2})\b", s)
    if m:
        return f"Quận {int(m.group(1))}"

    for key, val in sorted(DISTRICT_CANONICAL.items(), key=lambda x: -len(x[0])):
        if key in s:
            return val

    return "Không rõ"


def extract_ward(location_text: Any) -> str:
    s0 = clean_text(location_text)
    m = re.search(r"(Phường|P\.)\s*([^,\-]+)", s0, flags=re.IGNORECASE)
    if m:
        return clean_text(m.group(2))
    return "Không rõ"


def location_zone(district: str) -> str:
    if district in CENTRAL_DISTRICTS:
        return "central"
    if district in DEVELOPING_DISTRICTS:
        return "developing"
    if district in SUBURBAN_DISTRICTS:
        return "suburban"
    if district == "Không rõ":
        return "unknown"
    return "urban"


def infer_property_type(text_all: Any) -> str:
    s = norm_text(text_all)

    # Ưu tiên dấu hiệu mạnh trước để tránh nhầm "nhà mặt tiền gần chung cư" thành apartment.
    if any(k in s for k in ["dat nen", "lo dat", "dat "]):
        return "land"
    if any(k in s for k in ["biet thu", "villa"]):
        return "villa"
    if any(k in s for k in ["shophouse", "nha pho", "nha mat pho", "mat tien", "nha mt", "nha mat tien"]):
        return "townhouse"
    if any(k in s for k in ["nha rieng", "nha hem", "hem", "hxh", "nha "]):
        return "house"
    if any(k in s for k in ["can ho", "chung cu", "apartment", "vinhomes", "officetel"]):
        return "apartment"
    return "unknown"


def normalize_legal(raw_legal: Any, text_all: str = "") -> str:
    s = norm_text(str(raw_legal) + " " + str(text_all))
    if s.strip() == "":
        return "unknown"

    if any(k in s for k in ["so hong rieng", "shr", "so do", "so hong", "phap ly so", "so rieng"]):
        return "clear_ownership"
    if any(k in s for k in ["hop dong mua ban", "hdmb", "hop dong", "vi bang"]):
        return "contract_based"
    if any(k in s for k in ["dang cho so", "cho so", "chua so", "chua co so", "dang lam so"]):
        return "pending"
    if any(k in s for k in ["khong ro", "dang cap nhat", "nan"]):
        return "unknown"
    return "other"


def legal_score(legal_class: str) -> float:
    mapping = {
        "clear_ownership": 1.00,
        "contract_based": 0.60,
        "pending": 0.40,
        "other": 0.50,
        "unknown": 0.20,
    }
    return mapping.get(legal_class, 0.20)


AMENITY_PATTERNS = {
    "school": [r"truong hoc", r"truong quoc te", r"mam non", r"tieu hoc", r"thcs", r"thpt", r"dai hoc", r"cho con"],
    "hospital": [r"benh vien", r"phong kham", r"y te"],
    # Không dùng pattern "\bcho\b" vì dễ nhầm với cụm "cho con" trong truy vấn.
    "market": [r"gan cho", r"cho dan sinh", r"cho truyen thong", r"sieu thi", r"bach hoa", r"winmart", r"coopmart", r"bigc", r"aeon"],
    "park": [r"cong vien", r"cay xanh", r"noi bo xanh"],
    "mall": [r"trung tam thuong mai", r"vincom", r"lotte", r"mall", r"aeon"],
    "business": [r"kinh doanh", r"mat tien", r"shophouse", r"van phong", r"spa", r"cua hang", r"nha hang"],
    "rental": [r"cho thue", r"hd thue", r"hop dong thue", r"dong tien", r"khai thac thue"],
    "car_access": [r"xe hoi", r"oto", r"o to", r"hxh", r"hem xe hoi", r"duong rong"],
    "furnished": [r"noi that", r"full noi that", r"day du noi that", r"co ban"],
    "river_view": [r"view song", r"ven song", r"song sai gon", r"view kenh"],
    "urgent_sale": [r"ban gap", r"can tien", r"giam gia", r"cat lo", r"keo gia"],
    "planning_infra": [r"quy hoach", r"ha tang", r"metro", r"cao toc", r"san bay", r"thu thiem", r"vanh dai"],
    "security": [r"an ninh", r"bao ve", r"camera", r"khu dan tri", r"yên tĩnh", r"yen tinh"],
}


def extract_amenity_tags(text_all: Any) -> List[str]:
    s = norm_text(text_all)
    tags = []
    for tag, patterns in AMENITY_PATTERNS.items():
        if any(re.search(pat, s) for pat in patterns):
            tags.append(tag)
    return sorted(set(tags))


In [6]:
# Cell 6 - Chuẩn hóa toàn bộ bảng dữ liệu

def get_col(df: pd.DataFrame, col: str, default: Any = "") -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series([default] * len(df), index=df.index)


def standardize_dataframe(raw: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()

    out["title"] = get_col(raw, "Tieu_De").apply(clean_text)
    out["description"] = get_col(raw, "Mo_Ta").apply(clean_text)
    out["url"] = get_col(raw, "URL").apply(clean_text)

    out["raw_location"] = get_col(raw, "Vi_Tri").apply(clean_text)
    out["raw_area"] = get_col(raw, "Dien_Tich").apply(clean_text)
    out["raw_price"] = get_col(raw, "Gia").apply(clean_text)
    out["raw_amenities"] = get_col(raw, "Tien_Ich").apply(clean_text)
    out["raw_legal"] = get_col(raw, "Phap_Ly").apply(clean_text)
    out["raw_furniture"] = get_col(raw, "Noi_That").apply(clean_text)

    out["text_all"] = (
        out["title"] + " " + out["description"] + " " + out["raw_amenities"] + " " +
        out["raw_location"] + " " + out["raw_legal"] + " " + out["raw_furniture"]
    ).apply(clean_text)

    out["area_m2"] = [parse_area_m2(a, t) for a, t in zip(out["raw_area"], out["text_all"])]
    out["price_billion"] = [parse_price_billion(p, t, a) for p, t, a in zip(out["raw_price"], out["text_all"], out["area_m2"])]

    out["price_per_m2_million"] = np.where(
        (out["area_m2"].notna()) & (out["area_m2"] > 0) & (out["price_billion"].notna()),
        out["price_billion"] * 1000 / out["area_m2"],
        np.nan,
    )

    out["bedrooms"] = [
        parse_int_value(v, t, keywords=["so phong ngu", "phong ngu", "pn"])
        for v, t in zip(get_col(raw, "So_PN"), out["text_all"])
    ]
    out["bathrooms"] = [
        parse_int_value(v, t, keywords=["so phong tam", "ve sinh", "wc", "phong tam"])
        for v, t in zip(get_col(raw, "So_PT"), out["text_all"])
    ]
    out["floors"] = [
        parse_int_value(v, t, keywords=["so tang", "tang"])
        for v, t in zip(get_col(raw, "So_Tang"), out["text_all"])
    ]

    out["frontage_m"] = [
        parse_meter(v, t, mode="frontage")
        for v, t in zip(get_col(raw, "Mat_Tien"), out["text_all"])
    ]
    out["road_width_m"] = [
        parse_meter(v, t, mode="road")
        for v, t in zip(get_col(raw, "Duong_Vao"), out["text_all"])
    ]

    out["district"] = out["raw_location"].apply(extract_district)
    out["ward"] = out["raw_location"].apply(extract_ward)
    out["location_zone"] = out["district"].apply(location_zone)
    out["property_type"] = out["text_all"].apply(infer_property_type)
    out["legal_class"] = [normalize_legal(l, t) for l, t in zip(out["raw_legal"], out["text_all"])]
    out["legal_score"] = out["legal_class"].apply(legal_score)
    out["amenity_tags"] = out["text_all"].apply(extract_amenity_tags)

    # ID ổn định, không dựa vào source
    out["property_id"] = [f"BDS_{i+1:05d}" for i in range(len(out))]

    return out

bds = standardize_dataframe(raw_df)
print("Sau chuẩn hóa ban đầu:", bds.shape)
display(bds.head(5))


Sau chuẩn hóa ban đầu: (1707, 26)


,title,description,url,raw_location,raw_area,raw_price,raw_amenities,raw_legal,raw_furniture,text_all,area_m2,price_billion,price_per_m2_million,bedrooms,bathrooms,floors,frontage_m,road_width_m,district,ward,location_zone,property_type,legal_class,legal_score,amenity_tags,property_id
0,,,,Phan Huy Ích,Ngang 3.5m dài 15m,5.95,Diện Tích : Ngang 3.5m dài 15m\r\n; Hướng : Tây Nam\r\n; Vị Trí : Phan Huy Ích - P.12 - Gò Vâp - HCM\r\n; Đường : 3m...,,,Diện Tích : Ngang 3.5m dài 15m\r\n; Hướng : Tây Nam\r\n; Vị Trí : Phan Huy Ích - P.12 - Gò Vâp - HCM\r\n; Đường : 3m...,52.5,5.95,113.333333,NaN,NaN,NaN,3.5,3.0,Không rõ,Không rõ,unknown,unknown,other,0.5,[],BDS_00001
1,,,,ĐƯỜNG Bùi Quang Là,Chưa xác định,5.35,Căn hộ tiêu chuẩn,,,Căn hộ tiêu chuẩn ĐƯỜNG Bùi Quang Là,NaN,5.35,NaN,NaN,NaN,NaN,NaN,NaN,Không rõ,Không rõ,unknown,apartment,other,0.5,[],BDS_00002
2,,,,thông 6m khu bàn cờ.\r\n,16x29,28 tỷ,Căn hộ tiêu chuẩn,,,Căn hộ tiêu chuẩn thông 6m khu bàn cờ.\r\n,464.0,28.00,60.344828,NaN,NaN,NaN,NaN,NaN,Không rõ,Không rõ,unknown,apartment,other,0.5,[],BDS_00003
3,,,,Nguyễn Văn Quá Q12 cần bán gấp.\r\n,4x14m,"5,2 tỷ",bao sang tên; chính chủ; hẻm,,,bao sang tên; chính chủ; hẻm Nguyễn Văn Quá Q12 cần bán gấp.\r\n,56.0,5.20,92.857143,NaN,NaN,NaN,NaN,NaN,Quận 12,\r\n,suburban,house,other,0.5,[urgent_sale],BDS_00004
4,,,,8m xe tải,4m,9tỷ,chính chủ; sân thượng,,,chính chủ; sân thượng 8m xe tải,NaN,9.00,NaN,NaN,NaN,NaN,NaN,NaN,Không rõ,Không rõ,unknown,unknown,other,0.5,[],BDS_00005


In [7]:
# Cell 7 - Khử trùng lặp và kiểm tra chất lượng dữ liệu

def make_fingerprint(row: pd.Series) -> str:
    title = norm_text(row.get("title", ""))[:80]
    loc = norm_text(row.get("raw_location", ""))[:80]
    price = row.get("price_billion", np.nan)
    area = row.get("area_m2", np.nan)
    price_key = "NA" if pd.isna(price) else str(round(float(price), 2))
    area_key = "NA" if pd.isna(area) else str(round(float(area), 1))
    return f"{title}|{loc}|{price_key}|{area_key}"


def clean_and_deduplicate(df: pd.DataFrame) -> pd.DataFrame:
    temp = df.copy()
    temp["url_norm"] = temp["url"].apply(lambda x: norm_text(x))
    temp["fingerprint"] = temp.apply(make_fingerprint, axis=1)

    # Ưu tiên dedup theo URL nếu có URL
    has_url = temp[temp["url_norm"] != ""].drop_duplicates("url_norm", keep="first")
    no_url = temp[temp["url_norm"] == ""]
    temp = pd.concat([has_url, no_url], ignore_index=True)

    # Dedup mềm theo fingerprint
    temp = temp.drop_duplicates("fingerprint", keep="first").reset_index(drop=True)

    # Reset ID sau dedup
    temp["property_id"] = [f"BDS_{i+1:05d}" for i in range(len(temp))]
    return temp

bds = clean_and_deduplicate(bds)
print("Sau khử trùng lặp:", bds.shape)


def validate_numeric_range(row: pd.Series) -> List[str]:
    flags = []
    price = row["price_billion"]
    area = row["area_m2"]
    ppm = row["price_per_m2_million"]

    if pd.isna(price):
        flags.append("missing_price")
    elif price < 0.2 or price > 300:
        flags.append("abnormal_price")

    if pd.isna(area):
        flags.append("missing_area")
    elif area < 10 or area > 1000:
        flags.append("abnormal_area")

    if pd.isna(ppm):
        flags.append("missing_price_per_m2")
    elif ppm < 5 or ppm > 1000:
        flags.append("abnormal_price_per_m2")

    if row["district"] == "Không rõ":
        flags.append("missing_district")

    if row["legal_class"] == "unknown":
        flags.append("missing_legal")

    if row["property_type"] == "unknown":
        flags.append("unknown_property_type")

    return flags


def data_quality_score(flags: List[str]) -> float:
    penalty_map = {
        "missing_price": 0.25,
        "abnormal_price": 0.30,
        "missing_area": 0.20,
        "abnormal_area": 0.25,
        "missing_price_per_m2": 0.10,
        "abnormal_price_per_m2": 0.20,
        "missing_district": 0.20,
        "missing_legal": 0.15,
        "unknown_property_type": 0.05,
    }
    score = 1.0
    for f in flags:
        score -= penalty_map.get(f, 0.05)
    return max(0.0, round(score, 3))

bds["data_flags"] = bds.apply(validate_numeric_range, axis=1)
bds["data_quality_score"] = bds["data_flags"].apply(data_quality_score)

display(bds[["property_id", "title", "district", "price_billion", "area_m2", "price_per_m2_million", "legal_class", "property_type", "data_quality_score", "data_flags"]].head(10))


Sau khử trùng lặp: (330, 28)


,property_id,title,district,price_billion,area_m2,price_per_m2_million,legal_class,property_type,data_quality_score,data_flags
0,BDS_00001,"Không mua căn này thì mua căn nào ạ! 5,7x13,3m 3 tầng KDC Him Lam - ở + kinh doanh tuyệt vời",Thủ Đức,8.9,76.0,117.105263,clear_ownership,townhouse,1.00,[]
1,BDS_00002,"Nhà MT Lạc Long Quân-Q11. Ngang 6m x 15m, sổ vuông (nở hậu 6,8m), 3 tầng, CN: 95m2, vị trí đắc địa",Quận 11,30.0,95.0,315.789474,clear_ownership,townhouse,1.00,[]
2,BDS_00003,Bán căn góc công viên KDC Thủ Thiêm,Quận 2,26.0,97.5,266.666667,clear_ownership,unknown,0.95,[unknown_property_type]
3,BDS_00004,"Bán nhà mặt tiền Nguyễn Thị Sóc, Bà Điểm, TPHCM. 168m2 (6x27m) đang thuê 25tr/th sổ hồng riêng",Hóc Môn,16.5,168.0,98.214286,clear_ownership,townhouse,1.00,[]
4,BDS_00005,"Toà CHDV hầm + 7 tầng mặt đường Ngô Đức Kế. Chỉ 34,8 tỷ",Bình Thạnh,34.8,144.7,240.497581,clear_ownership,land,1.00,[]
5,BDS_00006,Nhà mặt tiền KDC Bình Hưng sát Pegasus Tạ Quang Bửu - 96m2 - 4 tầng chỉ 9.9tỷ,Bình Chánh,9.9,96.0,103.125000,clear_ownership,townhouse,1.00,[]
6,BDS_00007,"Góc 2 mặt tiền Trần Đình Xu, 4.1x18m, nhà 3 lầu HĐ thuê 60tr/th, giá 24.8 tỷ TL",Quận 1,24.8,72.8,340.659341,clear_ownership,townhouse,1.00,[]
7,BDS_00008,"Mặt tiền Cô Bắc Quận 1, DT 5.2x17.3m, nhà 4 lầu gần phố đi bộ, chung cư Manhattan, giá 35 tỷ TL",Quận 1,35.0,92.9,376.749193,clear_ownership,townhouse,1.00,[]
8,BDS_00009,"Mặt tiền Sư Vạn Hạnh 4x18m, 5 tầng HĐT 78tr/th ngay gần Vạn Hạnh Mall sầm uất, giá 27.8 tỷ TL",Quận 10,27.8,67.8,410.029499,clear_ownership,townhouse,1.00,[]
9,BDS_00010,"32.9 tỷ sở hữu mặt tiền khu K300, Quận Tân Bình, DT: 6.6x31m, KC: 4 tầng, thu nhập: 150 triệu/tháng",Tân Bình,32.9,202.0,162.871287,clear_ownership,townhouse,1.00,[]


In [8]:
# Cell 8 - Symbolic bands và fuzzy membership

def price_band(price: float) -> str:
    if pd.isna(price): return "unknown"
    if price < 3: return "very_low"
    if price < 5: return "low"
    if price < 10: return "medium"
    if price < 20: return "medium_high"
    return "high"


def area_band(area: float) -> str:
    if pd.isna(area): return "unknown"
    if area < 50: return "small"
    if area < 90: return "medium"
    if area < 150: return "large"
    return "very_large"


def bedrooms_band(bedrooms: float) -> str:
    if pd.isna(bedrooms): return "unknown"
    if bedrooms <= 1: return "studio_or_1br"
    if bedrooms == 2: return "2br"
    if bedrooms == 3: return "3br"
    return "4br_plus"


def road_band(road_width: float) -> str:
    if pd.isna(road_width): return "unknown"
    if road_width < 3: return "small_alley"
    if road_width < 5: return "motorbike_alley"
    if road_width < 8: return "car_access"
    return "wide_road"


def frontage_band(frontage: float) -> str:
    if pd.isna(frontage): return "unknown"
    if frontage < 3.5: return "narrow"
    if frontage < 5: return "standard"
    if frontage < 8: return "wide"
    return "very_wide"


def trapezoid(x: float, a: float, b: float, c: float, d: float) -> float:
    """Membership function hình thang."""
    if pd.isna(x): return 0.0
    if x <= a or x >= d: return 0.0
    if b <= x <= c: return 1.0
    if a < x < b: return (x - a) / (b - a)
    if c < x < d: return (d - x) / (d - c)
    return 0.0


def has_tag(tags: List[str], tag: str) -> bool:
    return isinstance(tags, list) and tag in tags

bds["price_band"] = bds["price_billion"].apply(price_band)
bds["area_band"] = bds["area_m2"].apply(area_band)
bds["bedrooms_band"] = bds["bedrooms"].apply(bedrooms_band)
bds["road_band"] = bds["road_width_m"].apply(road_band)
bds["frontage_band"] = bds["frontage_m"].apply(frontage_band)

# Median giá/m2 theo quận để đánh giá giá hợp lý.
global_median_ppm = bds["price_per_m2_million"].median()
bds["district_median_price_per_m2"] = bds.groupby("district")["price_per_m2_million"].transform("median").fillna(global_median_ppm)


def price_reasonable_score(row: pd.Series) -> float:
    ppm = row["price_per_m2_million"]
    med = row["district_median_price_per_m2"]
    if pd.isna(ppm) or pd.isna(med) or med <= 0:
        return 0.50
    ratio = ppm / med
    if ratio <= 0.85:
        return 1.00
    if ratio <= 1.00:
        return 0.90
    if ratio <= 1.20:
        return 0.70
    if ratio <= 1.50:
        return 0.40
    return 0.15

bds["price_reasonable_score"] = bds.apply(price_reasonable_score, axis=1)

display(bds[["property_id", "district", "price_billion", "price_band", "area_m2", "area_band", "bedrooms_band", "road_band", "frontage_band", "price_reasonable_score"]].head(10))


,property_id,district,price_billion,price_band,area_m2,area_band,bedrooms_band,road_band,frontage_band,price_reasonable_score
0,BDS_00001,Thủ Đức,8.9,medium,76.0,medium,3br,unknown,wide,1.00
1,BDS_00002,Quận 11,30.0,high,95.0,large,unknown,unknown,wide,0.40
2,BDS_00003,Quận 2,26.0,high,97.5,large,4br_plus,wide_road,unknown,0.15
3,BDS_00004,Hóc Môn,16.5,medium_high,168.0,very_large,4br_plus,unknown,wide,0.15
4,BDS_00005,Bình Thạnh,34.8,high,144.7,large,4br_plus,motorbike_alley,unknown,0.90
5,BDS_00006,Bình Chánh,9.9,medium,96.0,large,4br_plus,wide_road,standard,0.90
6,BDS_00007,Quận 1,24.8,high,72.8,medium,4br_plus,wide_road,standard,1.00
7,BDS_00008,Quận 1,35.0,high,92.9,large,4br_plus,wide_road,wide,1.00
8,BDS_00009,Quận 10,27.8,high,67.8,medium,4br_plus,unknown,standard,0.70
9,BDS_00010,Tân Bình,32.9,high,202.0,very_large,4br_plus,unknown,unknown,1.00


In [9]:
# Cell 9 - Điểm symbolic nâng cao: gia đình, kinh doanh, cho thuê, đầu tư, rủi ro

def family_suitability(row: pd.Series) -> float:
    score = 0.0
    if not pd.isna(row["bedrooms"]):
        if row["bedrooms"] >= 3: score += 0.25
        elif row["bedrooms"] == 2: score += 0.20
        elif row["bedrooms"] == 1: score += 0.08
    if not pd.isna(row["bathrooms"]):
        if row["bathrooms"] >= 2: score += 0.15
        else: score += 0.06
    if not pd.isna(row["area_m2"]):
        if row["area_m2"] >= 70: score += 0.20
        elif row["area_m2"] >= 50: score += 0.15
        else: score += 0.05
    if row["legal_score"] >= 0.8: score += 0.18
    if row["location_zone"] in ["central", "urban", "developing"]: score += 0.07
    if has_tag(row["amenity_tags"], "school"): score += 0.08
    if has_tag(row["amenity_tags"], "market"): score += 0.04
    if has_tag(row["amenity_tags"], "hospital"): score += 0.03
    if has_tag(row["amenity_tags"], "security"): score += 0.05
    return round(min(score, 1.0), 3)


def business_potential(row: pd.Series) -> float:
    score = 0.0
    if row["property_type"] in ["townhouse", "villa"]: score += 0.18
    if has_tag(row["amenity_tags"], "business"): score += 0.22
    if not pd.isna(row["frontage_m"]):
        if row["frontage_m"] >= 6: score += 0.22
        elif row["frontage_m"] >= 4: score += 0.16
        elif row["frontage_m"] >= 3.5: score += 0.08
    if not pd.isna(row["road_width_m"]):
        if row["road_width_m"] >= 8: score += 0.20
        elif row["road_width_m"] >= 5: score += 0.14
        elif row["road_width_m"] >= 3: score += 0.05
    if has_tag(row["amenity_tags"], "car_access"): score += 0.10
    if row["location_zone"] in ["central", "urban"]: score += 0.08
    return round(min(score, 1.0), 3)


def rental_potential(row: pd.Series) -> float:
    score = 0.0
    if row["property_type"] in ["apartment", "townhouse", "house"]: score += 0.15
    if has_tag(row["amenity_tags"], "rental"): score += 0.25
    if row["location_zone"] == "central": score += 0.22
    elif row["location_zone"] in ["urban", "developing"]: score += 0.16
    if not pd.isna(row["bedrooms"]):
        if 1 <= row["bedrooms"] <= 3: score += 0.12
        elif row["bedrooms"] >= 4: score += 0.08
    if row["legal_score"] >= 0.8: score += 0.12
    if has_tag(row["amenity_tags"], "furnished"): score += 0.06
    if has_tag(row["amenity_tags"], "mall") or has_tag(row["amenity_tags"], "market"): score += 0.05
    if row["price_reasonable_score"] >= 0.7: score += 0.05
    return round(min(score, 1.0), 3)


def investment_potential(row: pd.Series) -> float:
    score = 0.0
    if row["location_zone"] == "central": score += 0.25
    elif row["location_zone"] == "developing": score += 0.22
    elif row["location_zone"] == "urban": score += 0.16
    if row["legal_score"] >= 0.8: score += 0.18
    elif row["legal_score"] >= 0.6: score += 0.10
    score += 0.22 * row["price_reasonable_score"]
    if has_tag(row["amenity_tags"], "planning_infra"): score += 0.14
    if has_tag(row["amenity_tags"], "urgent_sale"): score += 0.06
    if row["property_type"] in ["townhouse", "apartment", "land"]: score += 0.08
    if row["data_quality_score"] >= 0.8: score += 0.07
    return round(min(score, 1.0), 3)


def risk_score(row: pd.Series) -> float:
    risk = 0.0
    if row["legal_class"] == "unknown": risk += 0.25
    elif row["legal_class"] in ["pending", "contract_based"]: risk += 0.15
    if row["data_quality_score"] < 0.6: risk += 0.25
    elif row["data_quality_score"] < 0.8: risk += 0.10
    if row["price_reasonable_score"] < 0.4: risk += 0.15
    if "abnormal_price_per_m2" in row["data_flags"]: risk += 0.20
    if "missing_district" in row["data_flags"]: risk += 0.10
    return round(min(risk, 1.0), 3)

bds["family_suitability_score"] = bds.apply(family_suitability, axis=1)
bds["business_potential_score"] = bds.apply(business_potential, axis=1)
bds["rental_potential_score"] = bds.apply(rental_potential, axis=1)
bds["investment_potential_score"] = bds.apply(investment_potential, axis=1)
bds["risk_score"] = bds.apply(risk_score, axis=1)


def symbolic_level(score: float) -> str:
    if pd.isna(score): return "unknown"
    if score >= 0.75: return "high"
    if score >= 0.50: return "medium"
    if score >= 0.25: return "low"
    return "very_low"

for col in [
    "family_suitability_score", "business_potential_score", "rental_potential_score",
    "investment_potential_score", "risk_score", "data_quality_score"
]:
    bds[col.replace("_score", "_level")] = bds[col].apply(symbolic_level)

display(bds[["property_id", "property_type", "district", "legal_class", "family_suitability_score", "business_potential_score", "rental_potential_score", "investment_potential_score", "risk_score", "data_quality_score"]].head(10))


,property_id,property_type,district,legal_class,family_suitability_score,business_potential_score,rental_potential_score,investment_potential_score,risk_score,data_quality_score
0,BDS_00001,townhouse,Thủ Đức,clear_ownership,0.85,0.66,0.60,0.770,0.00,1.00
1,BDS_00002,townhouse,Quận 11,clear_ownership,0.53,0.70,0.68,0.638,0.00,1.00
2,BDS_00003,unknown,Quận 2,clear_ownership,0.85,0.20,0.42,0.643,0.15,0.95
3,BDS_00004,townhouse,Hóc Môn,clear_ownership,0.90,0.62,0.71,0.363,0.15,1.00
4,BDS_00005,land,Bình Thạnh,clear_ownership,0.70,0.35,0.78,0.778,0.00,1.00
5,BDS_00006,townhouse,Bình Chánh,clear_ownership,0.90,0.76,0.81,0.888,0.00,1.00
6,BDS_00007,townhouse,Quận 1,clear_ownership,0.85,0.84,0.87,0.800,0.00,1.00
7,BDS_00008,townhouse,Quận 1,clear_ownership,0.85,0.84,0.62,0.800,0.00,1.00
8,BDS_00009,townhouse,Quận 10,clear_ownership,0.80,0.64,0.92,0.734,0.00,1.00
9,BDS_00010,townhouse,Tân Bình,clear_ownership,0.85,0.48,0.81,0.710,0.00,1.00


In [10]:
# Cell 10 - Xây tập luật Symbolic chặt hơn: có điều kiện, kết luận, strength, evidence

RULE_BASE = [
    {
        "rule_id": "R01_LEGAL_CLEAR",
        "target": "legal_risk",
        "conclusion": "low_legal_risk",
        "description": "Pháp lý rõ khi có sổ đỏ/sổ hồng/sổ hồng riêng.",
    },
    {
        "rule_id": "R02_LEGAL_WARNING",
        "target": "risk_warning",
        "conclusion": "need_legal_verification",
        "description": "Pháp lý không rõ, đang chờ sổ hoặc hợp đồng mua bán cần kiểm tra thêm.",
    },
    {
        "rule_id": "R03_FAMILY_STRONG",
        "target": "suitable_for",
        "conclusion": "family",
        "description": "Phù hợp gia đình nếu đủ phòng, diện tích hợp lý, pháp lý tốt và tiện ích sống tốt.",
    },
    {
        "rule_id": "R04_BUSINESS_STRONG",
        "target": "suitable_for",
        "conclusion": "business",
        "description": "Phù hợp kinh doanh nếu có mặt tiền/đường lớn/từ khóa kinh doanh.",
    },
    {
        "rule_id": "R05_RENTAL_STRONG",
        "target": "suitable_for",
        "conclusion": "rental",
        "description": "Phù hợp cho thuê nếu vị trí tốt, loại hình dễ cho thuê và có tín hiệu khai thác thuê.",
    },
    {
        "rule_id": "R06_INVESTMENT_STRONG",
        "target": "suitable_for",
        "conclusion": "investment",
        "description": "Phù hợp đầu tư nếu vị trí tốt, pháp lý rõ và giá/m2 hợp lý so với khu vực.",
    },
    {
        "rule_id": "R07_DATA_LOW_CONFIDENCE",
        "target": "risk_warning",
        "conclusion": "low_data_confidence",
        "description": "Dữ liệu thiếu nhiều trường quan trọng làm giảm độ tin cậy khuyến nghị.",
    },
    {
        "rule_id": "R08_OVERPRICED_WARNING",
        "target": "risk_warning",
        "conclusion": "possibly_overpriced",
        "description": "Giá/m2 cao hơn đáng kể so với trung vị khu vực.",
    },
]


def apply_rule_engine(row: pd.Series) -> Tuple[List[str], List[Dict[str, Any]], List[str]]:
    facts = []
    evidence = []
    risk_flags = []

    # R01
    if row["legal_class"] == "clear_ownership":
        facts.append("low_legal_risk")
        evidence.append({
            "rule_id": "R01_LEGAL_CLEAR",
            "strength": 1.0,
            "evidence": f"legal_class={row['legal_class']}",
        })

    # R02
    if row["legal_class"] in ["unknown", "pending", "contract_based", "other"]:
        strength = 0.9 if row["legal_class"] == "unknown" else 0.7
        risk_flags.append("need_legal_verification")
        evidence.append({
            "rule_id": "R02_LEGAL_WARNING",
            "strength": strength,
            "evidence": f"legal_class={row['legal_class']}",
        })

    # R03
    if row["family_suitability_score"] >= 0.70:
        facts.append("suitable_for_family")
        evidence.append({
            "rule_id": "R03_FAMILY_STRONG",
            "strength": float(row["family_suitability_score"]),
            "evidence": f"bedrooms={row['bedrooms']}, area={row['area_m2']}, legal_score={row['legal_score']}",
        })

    # R04
    if row["business_potential_score"] >= 0.65:
        facts.append("suitable_for_business")
        evidence.append({
            "rule_id": "R04_BUSINESS_STRONG",
            "strength": float(row["business_potential_score"]),
            "evidence": f"frontage={row['frontage_m']}, road={row['road_width_m']}, tags={row['amenity_tags']}",
        })

    # R05
    if row["rental_potential_score"] >= 0.65:
        facts.append("suitable_for_rental")
        evidence.append({
            "rule_id": "R05_RENTAL_STRONG",
            "strength": float(row["rental_potential_score"]),
            "evidence": f"zone={row['location_zone']}, type={row['property_type']}, tags={row['amenity_tags']}",
        })

    # R06
    if row["investment_potential_score"] >= 0.70:
        facts.append("suitable_for_investment")
        evidence.append({
            "rule_id": "R06_INVESTMENT_STRONG",
            "strength": float(row["investment_potential_score"]),
            "evidence": f"zone={row['location_zone']}, legal_score={row['legal_score']}, price_reasonable={row['price_reasonable_score']}",
        })

    # R07
    if row["data_quality_score"] < 0.70:
        risk_flags.append("low_data_confidence")
        evidence.append({
            "rule_id": "R07_DATA_LOW_CONFIDENCE",
            "strength": round(1 - row["data_quality_score"], 3),
            "evidence": f"data_flags={row['data_flags']}",
        })

    # R08
    if row["price_reasonable_score"] < 0.40:
        risk_flags.append("possibly_overpriced")
        evidence.append({
            "rule_id": "R08_OVERPRICED_WARNING",
            "strength": round(1 - row["price_reasonable_score"], 3),
            "evidence": f"price_per_m2={row['price_per_m2_million']}, district_median={row['district_median_price_per_m2']}",
        })

    return sorted(set(facts)), evidence, sorted(set(risk_flags))

rule_outputs = bds.apply(apply_rule_engine, axis=1)
bds["symbolic_facts"] = [x[0] for x in rule_outputs]
bds["rule_evidence"] = [x[1] for x in rule_outputs]
bds["risk_flags"] = [x[2] for x in rule_outputs]
bds["triggered_rules"] = [[ev["rule_id"] for ev in evs] for evs in bds["rule_evidence"]]

display(bds[["property_id", "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence"]].head(10))


,property_id,symbolic_facts,risk_flags,triggered_rules,rule_evidence
0,BDS_00001,"[low_legal_risk, suitable_for_business, suitable_for_family, suitable_for_investment]",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R04_BUSINESS_STRONG, R06_INVESTMENT_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
1,BDS_00002,"[low_legal_risk, suitable_for_business, suitable_for_rental]",[],"[R01_LEGAL_CLEAR, R04_BUSINESS_STRONG, R05_RENTAL_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R04_BUSINE..."
2,BDS_00003,"[low_legal_risk, suitable_for_family]",[possibly_overpriced],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R08_OVERPRICED_WARNING]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
3,BDS_00004,"[low_legal_risk, suitable_for_family, suitable_for_rental]",[possibly_overpriced],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R05_RENTAL_STRONG, R08_OVERPRICED_WARNING]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
4,BDS_00005,"[low_legal_risk, suitable_for_family, suitable_for_investment, suitable_for_rental]",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R05_RENTAL_STRONG, R06_INVESTMENT_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
5,BDS_00006,"[low_legal_risk, suitable_for_business, suitable_for_family, suitable_for_investment, suitable_for_rental]",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R04_BUSINESS_STRONG, R05_RENTAL_STRONG, R06_INVESTMENT_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
6,BDS_00007,"[low_legal_risk, suitable_for_business, suitable_for_family, suitable_for_investment, suitable_for_rental]",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R04_BUSINESS_STRONG, R05_RENTAL_STRONG, R06_INVESTMENT_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
7,BDS_00008,"[low_legal_risk, suitable_for_business, suitable_for_family, suitable_for_investment]",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R04_BUSINESS_STRONG, R06_INVESTMENT_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
8,BDS_00009,"[low_legal_risk, suitable_for_family, suitable_for_investment, suitable_for_rental]",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R05_RENTAL_STRONG, R06_INVESTMENT_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."
9,BDS_00010,"[low_legal_risk, suitable_for_family, suitable_for_investment, suitable_for_rental]",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R05_RENTAL_STRONG, R06_INVESTMENT_STRONG]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY..."


In [11]:
# Cell 11 - Ma trận tương đồng vị trí để hỗ trợ Spreading Activation

LOCATION_SIMILARITY_MANUAL = {
    ("Quận 3", "Quận 1"): 0.90,
    ("Quận 3", "Quận 10"): 0.85,
    ("Quận 3", "Phú Nhuận"): 0.75,
    ("Quận 1", "Bình Thạnh"): 0.75,
    ("Quận 1", "Quận 4"): 0.75,
    ("Quận 10", "Quận 5"): 0.80,
    ("Quận 10", "Quận 11"): 0.75,
    ("Bình Thạnh", "Phú Nhuận"): 0.75,
    ("Bình Thạnh", "Thủ Đức"): 0.60,
    ("Quận 2", "Thủ Đức"): 0.75,
    ("Quận 7", "Nhà Bè"): 0.75,
    ("Bình Tân", "Tân Phú"): 0.70,
    ("Tân Bình", "Phú Nhuận"): 0.75,
    ("Gò Vấp", "Phú Nhuận"): 0.65,
    ("Gò Vấp", "Tân Bình"): 0.65,
}


def district_similarity(a: str, b: str) -> float:
    if a == b:
        return 1.0
    if a == "Không rõ" or b == "Không rõ":
        return 0.15
    if (a, b) in LOCATION_SIMILARITY_MANUAL:
        return LOCATION_SIMILARITY_MANUAL[(a, b)]
    if (b, a) in LOCATION_SIMILARITY_MANUAL:
        return LOCATION_SIMILARITY_MANUAL[(b, a)]

    za, zb = location_zone(a), location_zone(b)
    if za == zb and za != "unknown":
        return 0.60
    if {za, zb} == {"central", "urban"}:
        return 0.50
    if {za, zb} == {"central", "developing"}:
        return 0.42
    if {za, zb} == {"developing", "urban"}:
        return 0.45
    if {za, zb} == {"developing", "suburban"}:
        return 0.35
    return 0.25


districts = sorted(bds["district"].dropna().unique().tolist())
location_similarity_matrix = pd.DataFrame(index=districts, columns=districts, dtype=float)
for a in districts:
    for b in districts:
        location_similarity_matrix.loc[a, b] = district_similarity(a, b)

display(location_similarity_matrix.head())


,Bình Chánh,Bình Thạnh,Bình Tân,Củ Chi,Gò Vấp,Hóc Môn,Không rõ,Nhà Bè,Phú Nhuận,Quận 1,Quận 10,Quận 11,Quận 12,Quận 2,Quận 3,Quận 4,Quận 5,Quận 6,Quận 7,Quận 8,Quận 9,Thủ Đức,Tân Bình,Tân Phú
Bình Chánh,1.00,0.42,0.35,0.35,0.45,0.35,0.15,0.60,0.42,0.42,0.42,0.45,0.35,0.60,0.42,0.45,0.42,0.45,0.60,0.45,0.60,0.60,0.45,0.45
Bình Thạnh,0.42,1.00,0.25,0.25,0.50,0.25,0.15,0.42,0.75,0.75,0.60,0.50,0.25,0.42,0.60,0.50,0.60,0.50,0.42,0.50,0.42,0.60,0.50,0.50
Bình Tân,0.35,0.25,1.00,0.60,0.25,0.60,0.15,0.35,0.25,0.25,0.25,0.25,0.60,0.35,0.25,0.25,0.25,0.25,0.35,0.25,0.35,0.35,0.25,0.70
Củ Chi,0.35,0.25,0.60,1.00,0.25,0.60,0.15,0.35,0.25,0.25,0.25,0.25,0.60,0.35,0.25,0.25,0.25,0.25,0.35,0.25,0.35,0.35,0.25,0.25
Gò Vấp,0.45,0.50,0.25,0.25,1.00,0.25,0.15,0.45,0.65,0.50,0.50,0.60,0.25,0.45,0.50,0.60,0.50,0.60,0.45,0.60,0.45,0.45,0.65,0.60


In [12]:
# Cell 12 - Sinh triples cho Knowledge Graph, không có source

def create_triples(df: pd.DataFrame) -> pd.DataFrame:
    triples = []
    numeric_predicates = {
        "price_billion": "hasPriceBillion",
        "area_m2": "hasAreaM2",
        "price_per_m2_million": "hasPricePerM2Million",
        "bedrooms": "hasBedrooms",
        "bathrooms": "hasBathrooms",
        "floors": "hasFloors",
        "frontage_m": "hasFrontageM",
        "road_width_m": "hasRoadWidthM",
        "legal_score": "hasLegalScore",
        "data_quality_score": "hasDataQualityScore",
        "risk_score": "hasRiskScore",
        "family_suitability_score": "hasFamilySuitabilityScore",
        "business_potential_score": "hasBusinessPotentialScore",
        "rental_potential_score": "hasRentalPotentialScore",
        "investment_potential_score": "hasInvestmentPotentialScore",
    }

    for _, row in df.iterrows():
        s = row["property_id"]
        triples.append([s, "type", "Property"])
        triples.append([s, "hasPropertyType", row["property_type"]])
        triples.append([s, "locatedInDistrict", row["district"]])
        triples.append([s, "locatedInWard", row["ward"]])
        triples.append([s, "locatedInZone", row["location_zone"]])
        triples.append([s, "hasPriceBand", row["price_band"]])
        triples.append([s, "hasAreaBand", row["area_band"]])
        triples.append([s, "hasBedroomBand", row["bedrooms_band"]])
        triples.append([s, "hasRoadBand", row["road_band"]])
        triples.append([s, "hasFrontageBand", row["frontage_band"]])
        triples.append([s, "hasLegalClass", row["legal_class"]])

        for col, pred in numeric_predicates.items():
            val = row.get(col, np.nan)
            if not pd.isna(val):
                triples.append([s, pred, float(val)])

        for tag in row["amenity_tags"]:
            triples.append([s, "hasAmenityTag", tag])

        for fact in row["symbolic_facts"]:
            triples.append([s, "inferredFact", fact])

        for flag in row["risk_flags"]:
            triples.append([s, "hasRiskFlag", flag])

        for rule in row["triggered_rules"]:
            triples.append([s, "triggeredRule", rule])

    return pd.DataFrame(triples, columns=["subject", "predicate", "object"])

kg_triples = create_triples(bds)
print("Số triples:", len(kg_triples))
display(kg_triples.head(25))


Số triples: 9823


,subject,predicate,object
0,BDS_00001,type,Property
1,BDS_00001,hasPropertyType,townhouse
2,BDS_00001,locatedInDistrict,Thủ Đức
3,BDS_00001,locatedInWard,Linh Chiểu
4,BDS_00001,locatedInZone,developing
5,BDS_00001,hasPriceBand,medium
6,BDS_00001,hasAreaBand,medium
7,BDS_00001,hasBedroomBand,3br
8,BDS_00001,hasRoadBand,unknown
9,BDS_00001,hasFrontageBand,wide


In [13]:
# Cell 13 - Parse truy vấn người dùng thành symbolic profile

def parse_user_query(query: str) -> Dict[str, Any]:
    q_raw = clean_text(query)
    q = norm_text(query)

    profile = {
        "raw_query": q_raw,
        "budget_billion": None,
        "target_area_m2": None,
        "district": None,
        "bedrooms_min": None,
        "bathrooms_min": None,
        "legal_required": False,
        "desired_amenities": [],
        "preferred_property_type": None,
        "purpose": "general",
        "hard_constraints": [],
        "soft_constraints": [],
    }

    # Ngân sách
    m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(ty|tỷ|ti|tỉ)", q_raw.lower())
    if m:
        profile["budget_billion"] = vn_float(m.group(1))
        profile["soft_constraints"].append("budget")

    # Diện tích
    m = re.search(r"(\d+(?:[\.,]\d+)?)\s*(m2|m²)", q_raw.lower())
    if m:
        profile["target_area_m2"] = vn_float(m.group(1))
        profile["soft_constraints"].append("area")

    # Số phòng ngủ / phòng tắm
    m = re.search(r"(\d+)\s*(pn|phong ngu|phòng ngủ)", q)
    if m:
        profile["bedrooms_min"] = int(m.group(1))
        profile["hard_constraints"].append("bedrooms_min")

    m = re.search(r"(\d+)\s*(wc|phong tam|phòng tắm|ve sinh)", q)
    if m:
        profile["bathrooms_min"] = int(m.group(1))
        profile["soft_constraints"].append("bathrooms_min")

    # Vị trí
    district = extract_district(q_raw)
    if district != "Không rõ":
        profile["district"] = district
        profile["soft_constraints"].append("location")

    # Pháp lý
    if any(k in q for k in ["phap ly ro", "so hong", "so do", "so rieng", "an toan phap ly", "pháp lý rõ"]):
        profile["legal_required"] = True
        profile["hard_constraints"].append("legal_required")

    # Loại BĐS
    if any(k in q for k in ["can ho", "chung cu", "apartment"]):
        profile["preferred_property_type"] = "apartment"
    elif any(k in q for k in ["nha pho", "mat tien", "mat pho", "shophouse"]):
        profile["preferred_property_type"] = "townhouse"
    elif any(k in q for k in ["biet thu", "villa"]):
        profile["preferred_property_type"] = "villa"
    elif any(k in q for k in ["nha rieng", "nha hem", "hxh"]):
        profile["preferred_property_type"] = "house"

    if profile["preferred_property_type"]:
        profile["soft_constraints"].append("property_type")

    # Mục đích
    if any(k in q for k in ["kinh doanh", "mat tien", "mo cua hang", "van phong", "shop"]):
        profile["purpose"] = "business"
    elif any(k in q for k in ["dau tu", "tang gia", "sinh loi", "tiem nang", "lướt sóng"]):
        profile["purpose"] = "investment"
    elif any(k in q for k in ["cho thue", "dong tien", "khai thac thue"]):
        profile["purpose"] = "rental"
    elif any(k in q for k in ["o", "gia dinh", "cho con", "an cu"]):
        profile["purpose"] = "family"

    # Tags tiện ích
    desired = []
    for tag, patterns in AMENITY_PATTERNS.items():
        if any(re.search(pat, q) for pat in patterns):
            desired.append(tag)
    profile["desired_amenities"] = sorted(set(desired))
    if desired:
        profile["soft_constraints"].append("amenity")

    return profile

example_query = "Tìm nhà quanh quận 3, khoảng 8 tỷ, 2PN trở lên, gần trường học cho con, pháp lý rõ ràng"
profile = parse_user_query(example_query)
print(json.dumps(profile, ensure_ascii=False, indent=2))


{
  "raw_query": "Tìm nhà quanh quận 3, khoảng 8 tỷ, 2PN trở lên, gần trường học cho con, pháp lý rõ ràng",
  "budget_billion": 8.0,
  "target_area_m2": null,
  "district": "Quận 3",
  "bedrooms_min": 2,
  "bathrooms_min": null,
  "legal_required": true,
  "desired_amenities": [
    "school"
  ],
  "preferred_property_type": null,
  "purpose": "family",
  "hard_constraints": [
    "bedrooms_min",
    "legal_required"
  ],
  "soft_constraints": [
    "budget",
    "location",
    "amenity"
  ]
}


In [14]:
# Cell 14 - Khuyến nghị bằng hard filter + fuzzy symbolic scoring

PURPOSE_SCORE_COL = {
    "family": "family_suitability_score",
    "business": "business_potential_score",
    "rental": "rental_potential_score",
    "investment": "investment_potential_score",
    "general": None,
}

WEIGHTS = {
    "family":      {"price": 0.24, "location": 0.22, "legal": 0.18, "area": 0.10, "amenity": 0.10, "type": 0.04, "purpose": 0.12},
    "business":    {"price": 0.18, "location": 0.22, "legal": 0.14, "area": 0.06, "amenity": 0.08, "type": 0.07, "purpose": 0.25},
    "rental":      {"price": 0.18, "location": 0.24, "legal": 0.14, "area": 0.07, "amenity": 0.08, "type": 0.07, "purpose": 0.22},
    "investment":  {"price": 0.24, "location": 0.20, "legal": 0.17, "area": 0.05, "amenity": 0.05, "type": 0.04, "purpose": 0.25},
    "general":     {"price": 0.23, "location": 0.22, "legal": 0.16, "area": 0.08, "amenity": 0.08, "type": 0.05, "purpose": 0.18},
}


def fuzzy_budget_score(price: float, budget: float | None) -> float:
    if budget is None or pd.isna(price):
        return 0.60
    if price <= budget:
        # Rẻ hơn ngân sách vẫn tốt, nhưng quá rẻ thì giảm nhẹ vì có thể khác phân khúc.
        ratio = price / budget
        if ratio >= 0.75: return 1.00
        if ratio >= 0.50: return 0.85
        return 0.70
    over = (price - budget) / budget
    if over <= 0.05: return 0.90
    if over <= 0.10: return 0.80
    if over <= 0.20: return 0.55
    if over <= 0.35: return 0.30
    return 0.05


def fuzzy_area_score(area: float, target_area: float | None) -> float:
    if target_area is None or pd.isna(area):
        return 0.60
    diff = abs(area - target_area) / max(target_area, 1)
    if diff <= 0.10: return 1.00
    if diff <= 0.25: return 0.80
    if diff <= 0.50: return 0.50
    return 0.20


def location_match_score(row_district: str, desired_district: str | None) -> float:
    if desired_district is None:
        return 0.60
    return district_similarity(row_district, desired_district)


def amenity_match_score(row_tags: List[str], desired_tags: List[str]) -> float:
    if not desired_tags:
        return 0.60
    if not isinstance(row_tags, list):
        return 0.0
    matched = len(set(row_tags) & set(desired_tags))
    return matched / len(set(desired_tags))


def property_type_score(row_type: str, preferred_type: str | None) -> float:
    if preferred_type is None:
        return 0.60
    if row_type == preferred_type:
        return 1.00
    # Gần nhau tương đối
    similar = {
        ("house", "townhouse"): 0.70,
        ("townhouse", "house"): 0.70,
        ("villa", "house"): 0.60,
        ("house", "villa"): 0.60,
    }
    return similar.get((row_type, preferred_type), 0.25)


def purpose_match_score(row: pd.Series, purpose: str) -> float:
    col = PURPOSE_SCORE_COL.get(purpose)
    if col is None:
        return max(row["family_suitability_score"], row["business_potential_score"], row["rental_potential_score"], row["investment_potential_score"])
    return row[col]


def hard_constraint_pass(row: pd.Series, profile: Dict[str, Any], relax_level: int = 0) -> bool:
    """
    relax_level = 0: chặt nhất
    relax_level = 1: nới ngân sách và phòng ngủ nhẹ
    relax_level = 2: nới pháp lý từ clear sang contract/pending nhưng sẽ bị phạt điểm
    """
    if profile.get("legal_required"):
        if relax_level < 2 and row["legal_class"] != "clear_ownership":
            return False
        if relax_level >= 2 and row["legal_score"] < 0.5:
            return False

    bedrooms_min = profile.get("bedrooms_min")
    if bedrooms_min is not None:
        # Chặt hơn bản cũ: nếu user yêu cầu số phòng ngủ mà tin đăng thiếu phòng ngủ,
        # relax_level 0 sẽ loại; relax_level >= 1 mới giữ lại nhưng điểm mục đích vẫn bị ảnh hưởng bởi thiếu dữ liệu.
        if pd.isna(row["bedrooms"]):
            if relax_level == 0:
                return False
        else:
            required = bedrooms_min if relax_level == 0 else max(1, bedrooms_min - 1)
            if row["bedrooms"] < required:
                return False

    return True


def score_property(row: pd.Series, profile: Dict[str, Any]) -> Tuple[float, Dict[str, float]]:
    detail = {}
    detail["price"] = fuzzy_budget_score(row["price_billion"], profile.get("budget_billion"))
    detail["location"] = location_match_score(row["district"], profile.get("district"))
    detail["legal"] = row["legal_score"] if profile.get("legal_required") else (0.65 + 0.35 * row["legal_score"])
    detail["area"] = fuzzy_area_score(row["area_m2"], profile.get("target_area_m2"))
    detail["amenity"] = amenity_match_score(row["amenity_tags"], profile.get("desired_amenities", []))
    detail["type"] = property_type_score(row["property_type"], profile.get("preferred_property_type"))
    detail["purpose"] = purpose_match_score(row, profile.get("purpose", "general"))

    weights = WEIGHTS.get(profile.get("purpose", "general"), WEIGHTS["general"])
    base = sum(detail[k] * weights[k] for k in weights)

    # Phạt rủi ro và dữ liệu thiếu.
    penalty = 1.0 - 0.22 * row["risk_score"]
    quality_boost = 0.85 + 0.15 * row["data_quality_score"]
    final = base * penalty * quality_boost

    return round(float(final), 4), {k: round(float(v), 4) for k, v in detail.items()}


def explain_row(row: pd.Series, profile: Dict[str, Any], detail: Dict[str, float]) -> str:
    parts = []
    if profile.get("district"):
        parts.append(f"Vị trí {row['district']} đạt điểm tương đồng {detail['location']:.2f} với {profile['district']}.")
    if profile.get("budget_billion") and not pd.isna(row["price_billion"]):
        parts.append(f"Giá {row['price_billion']:.2f} tỷ so với ngân sách {profile['budget_billion']:.2f} tỷ, điểm giá {detail['price']:.2f}.")
    if row["legal_class"] == "clear_ownership":
        parts.append("Pháp lý rõ, thuộc nhóm sổ đỏ/sổ hồng.")
    else:
        parts.append(f"Pháp lý thuộc nhóm {row['legal_class']}, cần kiểm tra thêm.")
    if detail["purpose"] >= 0.70:
        parts.append(f"Mục đích {profile.get('purpose')} phù hợp mạnh, điểm {detail['purpose']:.2f}.")
    elif detail["purpose"] >= 0.50:
        parts.append(f"Mục đích {profile.get('purpose')} phù hợp ở mức trung bình, điểm {detail['purpose']:.2f}.")
    if row["risk_flags"]:
        parts.append("Cảnh báo: " + ", ".join(row["risk_flags"]) + ".")
    if row["amenity_tags"]:
        parts.append("Tag nổi bật: " + ", ".join(row["amenity_tags"]) + ".")
    return " ".join(parts)


def recommend(query: str, df: pd.DataFrame, top_k: int = 5) -> Tuple[Dict[str, Any], pd.DataFrame]:
    profile = parse_user_query(query)

    # Thử từ chặt đến nới lỏng để tránh không có kết quả.
    candidate_df = None
    used_relax_level = 0
    for relax in [0, 1, 2]:
        mask = df.apply(lambda r: hard_constraint_pass(r, profile, relax_level=relax), axis=1)
        candidate_df = df[mask].copy()
        if len(candidate_df) > 0:
            used_relax_level = relax
            break

    results = []
    for _, row in candidate_df.iterrows():
        final, detail = score_property(row, profile)
        results.append({
            "property_id": row["property_id"],
            "final_score": final,
            "price_score": detail["price"],
            "location_score": detail["location"],
            "legal_score_used": detail["legal"],
            "area_score": detail["area"],
            "amenity_score": detail["amenity"],
            "type_score": detail["type"],
            "purpose_score": detail["purpose"],
            "property_type": row["property_type"],
            "title": row["title"],
            "district": row["district"],
            "ward": row["ward"],
            "price_billion": row["price_billion"],
            "area_m2": row["area_m2"],
            "price_per_m2_million": row["price_per_m2_million"],
            "bedrooms": row["bedrooms"],
            "bathrooms": row["bathrooms"],
            "legal_class": row["legal_class"],
            "amenity_tags": row["amenity_tags"],
            "symbolic_facts": row["symbolic_facts"],
            "risk_flags": row["risk_flags"],
            "triggered_rules": row["triggered_rules"],
            "data_quality_score": row["data_quality_score"],
            "explanation": explain_row(row, profile, detail),
            "url": row["url"],
        })

    out = pd.DataFrame(results)
    if len(out) > 0:
        out = out.sort_values("final_score", ascending=False).head(top_k).reset_index(drop=True)
    profile["used_relax_level"] = used_relax_level
    profile["num_candidates"] = len(candidate_df)
    return profile, out


In [15]:
# Cell 15 - Test truy vấn tư vấn BĐS

query = "Tìm nhà quanh quận 3, khoảng 8 tỷ, 2PN trở lên, gần trường học cho con, pháp lý rõ ràng"
profile, recs = recommend(query, bds, top_k=5)

print("QUERY:", query)
print("\nSYMBOLIC PROFILE:")
print(json.dumps(profile, ensure_ascii=False, indent=2))

print("\nTOP RESULTS:")
display(recs[[
    "property_id", "final_score", "property_type", "district", "price_billion", "area_m2",
    "bedrooms", "legal_class", "purpose_score", "risk_flags", "triggered_rules"
]])

for i, row in recs.iterrows():
    print("=" * 120)
    print(f"TOP {i+1} | {row['property_id']} | Score={row['final_score']}")
    print("Title:", row["title"])
    print("Explain:", row["explanation"])
    print("URL:", row["url"])


QUERY: Tìm nhà quanh quận 3, khoảng 8 tỷ, 2PN trở lên, gần trường học cho con, pháp lý rõ ràng

SYMBOLIC PROFILE:
{
  "raw_query": "Tìm nhà quanh quận 3, khoảng 8 tỷ, 2PN trở lên, gần trường học cho con, pháp lý rõ ràng",
  "budget_billion": 8.0,
  "target_area_m2": null,
  "district": "Quận 3",
  "bedrooms_min": 2,
  "bathrooms_min": null,
  "legal_required": true,
  "desired_amenities": [
    "school"
  ],
  "preferred_property_type": null,
  "purpose": "family",
  "hard_constraints": [
    "bedrooms_min",
    "legal_required"
  ],
  "soft_constraints": [
    "budget",
    "location",
    "amenity"
  ],
  "used_relax_level": 0,
  "num_candidates": 125
}

TOP RESULTS:


,property_id,final_score,property_type,district,price_billion,area_m2,bedrooms,legal_class,purpose_score,risk_flags,triggered_rules
0,BDS_00047,0.828,townhouse,Quận 6,8.00,66.3,2.0,clear_ownership,0.95,[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R05_RENTAL_STRONG]"
1,BDS_00143,0.761,townhouse,Quận 12,6.69,72.0,2.0,clear_ownership,0.85,[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R04_BUSINESS_STRONG]"
2,BDS_00206,0.751,apartment,Quận 10,5.10,72.0,2.0,clear_ownership,0.80,[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R05_RENTAL_STRONG, R06_INVESTMENT_STRONG]"
3,BDS_00144,0.725,townhouse,Quận 12,8.79,76.0,6.0,clear_ownership,0.95,[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R04_BUSINESS_STRONG]"
4,BDS_00112,0.710,townhouse,Tân Phú,7.90,53.1,2.0,clear_ownership,0.80,[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R05_RENTAL_STRONG]"


TOP 1 | BDS_00047 | Score=0.828
Title: Mặt tiền lô H Cư Xá Phú Lâm B Quận 6 | DT 4 x 18m, trệt + lầu, giá chỉ 8 tỷ
Explain: Vị trí Quận 6 đạt điểm tương đồng 0.50 với Quận 3. Giá 8.00 tỷ so với ngân sách 8.00 tỷ, điểm giá 1.00. Pháp lý rõ, thuộc nhóm sổ đỏ/sổ hồng. Mục đích family phù hợp mạnh, điểm 0.95. Tag nổi bật: business, hospital, market, school, security.
URL: https://batdongsan.com.vn/ban-nha-mat-pho-duong-cu-xa-phu-lam-b-phuong-13-3-58/tien-lo-h-quan-6-dt-4-x-18m-tret-lau-gia-chi-8-ty-pr45806465
TOP 2 | BDS_00143 | Score=0.761
Title: Nhà SHR 4m x 19m. Mặt tiền đường Nguyễn Thị Thơi (Hiệp Thành 18)
Explain: Vị trí Quận 12 đạt điểm tương đồng 0.25 với Quận 3. Giá 6.69 tỷ so với ngân sách 8.00 tỷ, điểm giá 1.00. Pháp lý rõ, thuộc nhóm sổ đỏ/sổ hồng. Mục đích family phù hợp mạnh, điểm 0.85. Tag nổi bật: business, market, planning_infra, school.
URL: https://batdongsan.com.vn/ban-nha-mat-pho-duong-nguyen-thi-thoi-phuong-hiep-thanh-3-64/shr-4m-x-19m-tien-uong-18-pr45496644
TOP 3 | 

In [16]:
# Cell 16 - Module so sánh 2 BĐS dựa trên Symbolic factors

def compare_properties(property_id_1: str, property_id_2: str, df: pd.DataFrame = bds) -> pd.DataFrame:
    a = df[df["property_id"] == property_id_1]
    b = df[df["property_id"] == property_id_2]
    if len(a) == 0:
        raise ValueError(f"Không tìm thấy {property_id_1}")
    if len(b) == 0:
        raise ValueError(f"Không tìm thấy {property_id_2}")
    a = a.iloc[0]
    b = b.iloc[0]

    fields = [
        ("Loại BĐS", "property_type"),
        ("Quận", "district"),
        ("Phường", "ward"),
        ("Nhóm vị trí", "location_zone"),
        ("Giá tỷ", "price_billion"),
        ("Diện tích m2", "area_m2"),
        ("Giá triệu/m2", "price_per_m2_million"),
        ("Số phòng ngủ", "bedrooms"),
        ("Số phòng tắm", "bathrooms"),
        ("Số tầng", "floors"),
        ("Mặt tiền m", "frontage_m"),
        ("Đường vào m", "road_width_m"),
        ("Pháp lý", "legal_class"),
        ("Điểm gia đình", "family_suitability_score"),
        ("Điểm kinh doanh", "business_potential_score"),
        ("Điểm cho thuê", "rental_potential_score"),
        ("Điểm đầu tư", "investment_potential_score"),
        ("Điểm rủi ro", "risk_score"),
        ("Độ tin cậy dữ liệu", "data_quality_score"),
        ("Symbolic facts", "symbolic_facts"),
        ("Risk flags", "risk_flags"),
    ]

    rows = []
    for label, col in fields:
        rows.append({
            "Tiêu chí": label,
            property_id_1: a[col],
            property_id_2: b[col],
        })
    return pd.DataFrame(rows)

if len(recs) >= 2:
    comparison = compare_properties(recs.loc[0, "property_id"], recs.loc[1, "property_id"])
    display(comparison)


,Tiêu chí,BDS_00047,BDS_00143
0,Loại BĐS,townhouse,townhouse
1,Quận,Quận 6,Quận 12
2,Phường,13,Hiệp Thành
3,Nhóm vị trí,urban,suburban
4,Giá tỷ,8.0,6.69
5,Diện tích m2,66.3,72.0
6,Giá triệu/m2,120.66365,92.916667
7,Số phòng ngủ,2.0,2.0
8,Số phòng tắm,2.0,2.0
9,Số tầng,NaN,NaN


In [17]:
# Cell 17 - Lưu dataset symbolic, rules, triples, matrix, kết quả mẫu
# Lưu ý: Không có cột source/source_name/source_file/source_sheet trong output.

EXPORT_COLUMNS = [
    "property_id",
    "title", "description", "url",
    "raw_location", "ward", "district", "location_zone", "property_type",
    "raw_price", "price_billion", "price_band",
    "raw_area", "area_m2", "area_band", "price_per_m2_million", "district_median_price_per_m2", "price_reasonable_score",
    "bedrooms", "bathrooms", "floors", "bedrooms_band",
    "frontage_m", "frontage_band", "road_width_m", "road_band",
    "legal_class", "legal_score",
    "amenity_tags",
    "family_suitability_score", "family_suitability_level",
    "business_potential_score", "business_potential_level",
    "rental_potential_score", "rental_potential_level",
    "investment_potential_score", "investment_potential_level",
    "risk_score", "risk_level",
    "data_quality_score", "data_quality_level", "data_flags",
    "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence",
]

# Chỉ lấy cột tồn tại
EXPORT_COLUMNS = [c for c in EXPORT_COLUMNS if c in bds.columns]

bds_export = bds[EXPORT_COLUMNS].copy()

# List/dict -> JSON string để CSV/XLSX dễ đọc
for col in ["amenity_tags", "data_flags", "symbolic_facts", "risk_flags", "triggered_rules", "rule_evidence"]:
    if col in bds_export.columns:
        bds_export[col] = bds_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))

bds_export.to_csv(OUTPUT_DIR / "bds_clean_symbolic_v2.csv", index=False, encoding="utf-8-sig")
bds_export.to_excel(OUTPUT_DIR / "bds_clean_symbolic_v2.xlsx", index=False)

with open(OUTPUT_DIR / "symbolic_rule_base_v2.json", "w", encoding="utf-8") as f:
    json.dump(RULE_BASE, f, ensure_ascii=False, indent=2)

kg_triples.to_csv(OUTPUT_DIR / "bds_kg_triples_v2.csv", index=False, encoding="utf-8-sig")
location_similarity_matrix.to_csv(OUTPUT_DIR / "location_similarity_matrix_v2.csv", encoding="utf-8-sig")

if 'recs' in globals() and len(recs) > 0:
    recs_export = recs.copy()
    for col in ["amenity_tags", "symbolic_facts", "risk_flags", "triggered_rules"]:
        if col in recs_export.columns:
            recs_export[col] = recs_export[col].apply(lambda x: json.dumps(x, ensure_ascii=False))
    recs_export.to_csv(OUTPUT_DIR / "sample_recommendation_results_v2.csv", index=False, encoding="utf-8-sig")
    recs_export.to_excel(OUTPUT_DIR / "sample_recommendation_results_v2.xlsx", index=False)

print("Đã lưu output:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print("-", p)


Đã lưu output:
- /content/bds_symbolic_output_v2/bds_clean_symbolic_v2.csv
- /content/bds_symbolic_output_v2/bds_clean_symbolic_v2.xlsx
- /content/bds_symbolic_output_v2/bds_kg_triples_v2.csv
- /content/bds_symbolic_output_v2/location_similarity_matrix_v2.csv
- /content/bds_symbolic_output_v2/sample_recommendation_results_v2.csv
- /content/bds_symbolic_output_v2/sample_recommendation_results_v2.xlsx
- /content/bds_symbolic_output_v2/symbolic_rule_base_v2.json


In [18]:
# Cell 18 - Nén output để tải về trong Colab
import shutil

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=str(OUTPUT_DIR))
print("File zip output:", zip_path)

# Nếu chạy trên Colab, có thể tải bằng:
# from google.colab import files
# files.download(zip_path)


File zip output: /content/bds_symbolic_output_v2.zip


#Visualization dataset

In [29]:
!pip -q install plotly openpyxl

import os
import ast
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
from IPython.display import display, HTML

In [30]:
# ================================
# CELL 2 - LOAD DỮ LIỆU SYMBOLIC SAU NOTEBOOK
# ================================

if "bds" in globals():
    df_viz = bds.copy()
    print("Đã dùng biến bds có sẵn trong notebook.")
elif "bds_export" in globals():
    df_viz = bds_export.copy()
    print("Đã dùng biến bds_export có sẵn trong notebook.")
else:
    possible_paths = [
        "/content/bds_symbolic_output/bds_clean_symbolic_v2.xlsx",
        "/content/bds_symbolic_output/bds_clean_symbolic_v2.csv",
        "/content/bds_clean_symbolic_v2.xlsx",
        "/content/bds_clean_symbolic_v2.csv"
    ]

    found_path = None
    for path in possible_paths:
        if os.path.exists(path):
            found_path = path
            break

    if found_path is None:
        raise FileNotFoundError("Không tìm thấy file bds_clean_symbolic_v2. Hãy chạy notebook chuẩn hóa trước.")

    if found_path.endswith(".xlsx"):
        df_viz = pd.read_excel(found_path)
    else:
        df_viz = pd.read_csv(found_path)

    print("Đã load dữ liệu từ:", found_path)

print("Kích thước dữ liệu:", df_viz.shape)
display(df_viz.head())

Đã dùng biến bds có sẵn trong notebook.
Kích thước dữ liệu: (330, 52)


,title,description,url,raw_location,raw_area,raw_price,raw_amenities,raw_legal,raw_furniture,text_all,area_m2,price_billion,price_per_m2_million,bedrooms,bathrooms,floors,frontage_m,road_width_m,district,ward,location_zone,property_type,legal_class,legal_score,amenity_tags,property_id,url_norm,fingerprint,data_flags,data_quality_score,price_band,area_band,bedrooms_band,road_band,frontage_band,district_median_price_per_m2,price_reasonable_score,family_suitability_score,business_potential_score,rental_potential_score,investment_potential_score,risk_score,family_suitability_level,business_potential_level,rental_potential_level,investment_potential_level,risk_level,data_quality_level,symbolic_facts,rule_evidence,risk_flags,triggered_rules
0,"Không mua căn này thì mua căn nào ạ! 5,7x13,3m 3 tầng KDC Him Lam - ở + kinh doanh tuyệt vời",Mã 213LC. Cần bán vội nên mới có giá này - KDC này kiếm không có căn ạ! Anh chị nhanh tay liên hệ em đi xem! Nhà mặt...,https://batdongsan.com.vn/ban-nha-mat-pho-phuong-linh-chieu/khong-mua-can-nay-thi-mua-can-nao-a-5-7x13-5m-3-tang-kdc...,"Phường Linh Chiểu, Thành phố Thủ Đức, Hồ Chí Minh",76 m²,"8,9 tỷ","Số phòng ngủ: 3 phòng | Số phòng tắm, vệ sinh: 4 phòng | Số tầng: 3 tầng | Mặt tiền: 5,7 m | Mức giá điện: Thỏa thuậ...",Sổ đỏ/ Sổ hồng.,,"Không mua căn này thì mua căn nào ạ! 5,7x13,3m 3 tầng KDC Him Lam - ở + kinh doanh tuyệt vời Mã 213LC. Cần bán vội n...",76.0,8.9,117.105263,3.0,4.0,3.0,5.7,NaN,Thủ Đức,Linh Chiểu,developing,townhouse,clear_ownership,1.0,"[business, car_access]",BDS_00001,https://batdongsan.com.vn/ban-nha-mat-pho-phuong-linh-chieu/khong-mua-can-nay-thi-mua-can-nao-a-5-7x13-5m-3-tang-kdc...,"khong mua can nay thi mua can nao a! 5,7x13,3m 3 tang kdc him lam - o + kinh doa|phuong linh chieu, thanh pho thu du...",[],1.00,medium,medium,3br,unknown,wide,176.650795,1.00,0.85,0.66,0.60,0.770,0.00,high,medium,medium,high,very_low,high,"[low_legal_risk, suitable_for_business, suitable_for_family, suitable_for_investment]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R03_FAMILY...",[],"[R01_LEGAL_CLEAR, R03_FAMILY_STRONG, R04_BUSINESS_STRONG, R06_INVESTMENT_STRONG]"
1,"Nhà MT Lạc Long Quân-Q11. Ngang 6m x 15m, sổ vuông (nở hậu 6,8m), 3 tầng, CN: 95m2, vị trí đắc địa","Cần tiền nên bán gấp - nhà mặt tiền Lạc Long Quân - nở hậu đẹp, Quận 11. Địa chỉ: MT Lạc Long Quân, Phường 10, Quận ...",https://batdongsan.com.vn/ban-nha-mat-pho-duong-lac-long-quan-phuong-10-7-63/mt-q11-ngang-6m-x-15m-so-vuong-no-hau-6...,"Đường Lạc Long Quân, Phường 10, Quận 11, Hồ Chí Minh",95 m²,30 tỷ,Số tầng: 3 tầng | Pháp lý: Sổ đỏ/ Sổ hồng,Sổ đỏ/ Sổ hồng,,"Nhà MT Lạc Long Quân-Q11. Ngang 6m x 15m, sổ vuông (nở hậu 6,8m), 3 tầng, CN: 95m2, vị trí đắc địa Cần tiền nên bán ...",95.0,30.0,315.789474,NaN,NaN,3.0,6.0,NaN,Quận 11,10,urban,townhouse,clear_ownership,1.0,"[business, rental, school, urgent_sale]",BDS_00002,https://batdongsan.com.vn/ban-nha-mat-pho-duong-lac-long-quan-phuong-10-7-63/mt-q11-ngang-6m-x-15m-so-vuong-no-hau-6...,"nha mt lac long quan-q11. ngang 6m x 15m, so vuong (no hau 6,8m), 3 tang, cn: 95|duong lac long quan, phuong 10, qua...",[],1.00,high,large,unknown,unknown,wide,218.181818,0.40,0.53,0.70,0.68,0.638,0.00,medium,medium,medium,medium,very_low,high,"[low_legal_risk, suitable_for_business, suitable_for_rental]","[{'rule_id': 'R01_LEGAL_CLEAR', 'strength': 1.0, 'evidence': 'legal_class=clear_ownership'}, {'rule_id': 'R04_BUSINE...",[],"[R01_LEGAL_CLEAR, R04_BUSINESS_STRONG, R05_RENTAL_STRONG]"
2,Bán căn góc công viên KDC Thủ Thiêm,"Bán căn góc trục chính Đông Thủ Thiêm Quận 2. - Địa chỉ: 4 đường 56 Bình Trung Đông Q2. - DT: 5x20m 97,5m² xây dựng ...",https://batdongsan.com.vn/ban-nha-mat-pho-duong-so-56-phuong-binh-trung-dong-khu-dan-cu-dong-thu-thiem/ban-can-goc-c...,"Khu dân cư Đông Thủ Thiêm, Đường Số 56, Phường Bình Trưng Đông, Quận 2, Hồ Chí Minh","97,5 m²",26 tỷ,"Số phòng ngủ: 5 phòng | Số phòng tắm, vệ sinh: 4 phòng | Số tầng: 4 t

In [31]:
# ================================
# CELL 3 - CHUẨN HÓA KIỂU LIST/DICT CHO CÁC CỘT SYMBOLIC
# ================================

def parse_json_or_list(x):
    if isinstance(x, list):
        return x

    if isinstance(x, dict):
        return [x]

    if pd.isna(x):
        return []

    s = str(x).strip()

    if s == "" or s.lower() in ["nan", "none", "null"]:
        return []

    try:
        val = json.loads(s)
        if isinstance(val, list):
            return val
        if isinstance(val, dict):
            return [val]
        return [val]
    except:
        pass

    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
        if isinstance(val, dict):
            return [val]
        return [val]
    except:
        pass

    if "," in s:
        return [i.strip() for i in s.split(",") if i.strip()]

    return [s]


list_cols = [
    "amenity_tags",
    "data_flags",
    "symbolic_facts",
    "risk_flags",
    "triggered_rules",
    "rule_evidence"
]

for col in list_cols:
    if col in df_viz.columns:
        df_viz[col] = df_viz[col].apply(parse_json_or_list)

numeric_cols = [
    "price_billion",
    "area_m2",
    "price_per_m2_million",
    "bedrooms",
    "bathrooms",
    "floors",
    "frontage_m",
    "road_width_m",
    "legal_score",
    "price_reasonable_score",
    "family_suitability_score",
    "business_potential_score",
    "rental_potential_score",
    "investment_potential_score",
    "risk_score",
    "data_quality_score"
]

for col in numeric_cols:
    if col in df_viz.columns:
        df_viz[col] = pd.to_numeric(df_viz[col], errors="coerce")

print("Các cột hiện có:")
print(df_viz.columns.tolist())

Các cột hiện có:
['title', 'description', 'url', 'raw_location', 'raw_area', 'raw_price', 'raw_amenities', 'raw_legal', 'raw_furniture', 'text_all', 'area_m2', 'price_billion', 'price_per_m2_million', 'bedrooms', 'bathrooms', 'floors', 'frontage_m', 'road_width_m', 'district', 'ward', 'location_zone', 'property_type', 'legal_class', 'legal_score', 'amenity_tags', 'property_id', 'url_norm', 'fingerprint', 'data_flags', 'data_quality_score', 'price_band', 'area_band', 'bedrooms_band', 'road_band', 'frontage_band', 'district_median_price_per_m2', 'price_reasonable_score', 'family_suitability_score', 'business_potential_score', 'rental_potential_score', 'investment_potential_score', 'risk_score', 'family_suitability_level', 'business_potential_level', 'rental_potential_level', 'investment_potential_level', 'risk_level', 'data_quality_level', 'symbolic_facts', 'rule_evidence', 'risk_flags', 'triggered_rules']


In [32]:
# ================================
# CELL 6 - PHÂN BỐ GIÁ, DIỆN TÍCH, GIÁ/M²
# ================================

if "price_billion" in df_viz.columns:
    fig = px.histogram(
        df_viz,
        x="price_billion",
        nbins=40,
        marginal="box",
        title="Phân bố giá BĐS"
    )
    fig.update_layout(xaxis_title="Giá, đơn vị tỷ VNĐ", yaxis_title="Số lượng")
    fig.show()

if "area_m2" in df_viz.columns:
    fig = px.histogram(
        df_viz,
        x="area_m2",
        nbins=40,
        marginal="box",
        title="Phân bố diện tích BĐS"
    )
    fig.update_layout(xaxis_title="Diện tích, m²", yaxis_title="Số lượng")
    fig.show()

if "price_per_m2_million" in df_viz.columns:
    fig = px.histogram(
        df_viz,
        x="price_per_m2_million",
        nbins=40,
        marginal="box",
        title="Phân bố giá trên mỗi m²"
    )
    fig.update_layout(xaxis_title="Giá/m², triệu VNĐ", yaxis_title="Số lượng")
    fig.show()

In [33]:
# ================================
# CELL 7 - THỐNG KÊ THEO QUẬN/HUYỆN VÀ NHÓM VỊ TRÍ
# ================================

if "district" in df_viz.columns:
    district_count = (
        df_viz["district"]
        .fillna("Không rõ")
        .value_counts()
        .reset_index()
    )
    district_count.columns = ["district", "count"]

    fig = px.bar(
        district_count.head(25),
        x="count",
        y="district",
        orientation="h",
        text="count",
        title="Top quận/huyện có nhiều BĐS nhất"
    )
    fig.update_layout(height=750, xaxis_title="Số lượng", yaxis_title="Quận/Huyện")
    fig.show()

    display(district_count)

if "location_zone" in df_viz.columns:
    zone_count = (
        df_viz["location_zone"]
        .fillna("unknown")
        .value_counts()
        .reset_index()
    )
    zone_count.columns = ["location_zone", "count"]

    fig = px.pie(
        zone_count,
        names="location_zone",
        values="count",
        title="Tỷ lệ BĐS theo nhóm vị trí Symbolic"
    )
    fig.show()

    display(zone_count)

,district,count
0,Không rõ,93
1,Quận 1,34
2,Quận 12,27
3,Gò Vấp,22
4,Tân Bình,20
5,Quận 10,20
6,Phú Nhuận,16
7,Tân Phú,16
8,Quận 3,13
9,Bình Tân,12


,location_zone,count
0,central,94
1,unknown,93
2,urban,68
3,suburban,43
4,developing,32


In [34]:
# ================================
# CELL 8 - THỐNG KÊ GIÁ TRUNG BÌNH THEO QUẬN/HUYỆN
# ================================

if {"district", "price_billion"}.issubset(df_viz.columns):
    district_price = (
        df_viz
        .dropna(subset=["district", "price_billion"])
        .groupby("district")
        .agg(
            count=("property_id", "count"),
            mean_price_billion=("price_billion", "mean"),
            median_price_billion=("price_billion", "median")
        )
        .reset_index()
        .sort_values("median_price_billion", ascending=False)
    )

    fig = px.bar(
        district_price.head(20),
        x="median_price_billion",
        y="district",
        orientation="h",
        text="count",
        title="Top quận/huyện theo giá trung vị"
    )
    fig.update_layout(
        height=700,
        xaxis_title="Giá trung vị, tỷ VNĐ",
        yaxis_title="Quận/Huyện"
    )
    fig.show()

    display(district_price)

if {"district", "price_per_m2_million"}.issubset(df_viz.columns):
    district_ppm = (
        df_viz
        .dropna(subset=["district", "price_per_m2_million"])
        .groupby("district")
        .agg(
            count=("property_id", "count"),
            mean_price_per_m2=("price_per_m2_million", "mean"),
            median_price_per_m2=("price_per_m2_million", "median")
        )
        .reset_index()
        .sort_values("median_price_per_m2", ascending=False)
    )

    fig = px.bar(
        district_ppm.head(20),
        x="median_price_per_m2",
        y="district",
        orientation="h",
        text="count",
        title="Top quận/huyện theo giá/m² trung vị"
    )
    fig.update_layout(
        height=700,
        xaxis_title="Giá/m² trung vị, triệu VNĐ",
        yaxis_title="Quận/Huyện"
    )
    fig.show()

    display(district_ppm)

,district,count,mean_price_billion,median_price_billion
9,Quận 1,34,58.617647,40.000
18,Quận 7,5,26.820000,33.800
10,Quận 10,20,56.115000,33.000
14,Quận 3,13,70.607692,33.000
22,Tân Bình,20,35.190000,23.000
16,Quận 5,3,25.333333,22.000
1,Bình Thạnh,8,31.660000,20.290
8,Phú Nhuận,16,22.387500,17.750
23,Tân Phú,16,25.621875,16.225
13,Quận 2,8,33.143750,16.000


,district,count,mean_price_per_m2,median_price_per_m2
14,Quận 3,13,473.596734,476.190476
9,Quận 1,34,586.219690,468.360715
10,Quận 10,20,383.052595,391.725908
16,Quận 5,3,321.595985,329.256820
8,Phú Nhuận,16,288.825816,293.560660
1,Bình Thạnh,8,268.425883,268.199234
22,Tân Bình,20,255.834174,251.262626
18,Quận 7,5,262.251605,247.126437
15,Quận 4,2,243.055556,243.055556
19,Quận 8,1,233.333333,233.333333


In [35]:
# ================================
# CELL 9 - PHÂN BỐ NHÓM SYMBOLIC: GIÁ, DIỆN TÍCH, LOẠI BĐS, PHÁP LÝ
# ================================

category_cols = [
    "price_band",
    "area_band",
    "bedrooms_band",
    "road_band",
    "frontage_band",
    "property_type",
    "legal_class",
    "risk_level",
    "data_quality_level",
    "family_suitability_level",
    "business_potential_level",
    "rental_potential_level",
    "investment_potential_level"
]

for col in category_cols:
    if col in df_viz.columns:
        temp = (
            df_viz[col]
            .fillna("unknown")
            .value_counts()
            .reset_index()
        )
        temp.columns = [col, "count"]

        fig = px.bar(
            temp,
            x=col,
            y="count",
            text="count",
            title=f"Phân bố {col}"
        )
        fig.update_layout(xaxis_title=col, yaxis_title="Số lượng")
        fig.show()

        display(temp)

,price_band,count
0,high,105
1,medium,83
2,medium_high,83
3,low,36
4,unknown,16
5,very_low,7


,area_band,count
0,medium,134
1,large,69
2,small,52
3,very_large,44
4,unknown,31


,bedrooms_band,count
0,unknown,189
1,4br_plus,96
2,2br,24
3,3br,15
4,studio_or_1br,6


,road_band,count
0,unknown,211
1,wide_road,107
2,car_access,8
3,motorbike_alley,4


,frontage_band,count
0,unknown,184
1,standard,68
2,wide,46
3,very_wide,27
4,narrow,5


...

,data_quality_level,count
0,high,285
1,medium,30
2,low,15


,family_suitability_level,count
0,very_low,104
1,high,94
2,low,84
3,medium,48


,business_potential_level,count
0,very_low,105
1,medium,80
2,low,74
3,high,71


,rental_potential_level,count
0,medium,98
1,very_low,98
2,low,76
3,high,58


,investment_potential_level,count
0,medium,125
1,low,82
2,high,66
3,very_low,57


In [36]:
# ================================
# CELL 10 - SCATTER GIÁ - DIỆN TÍCH
# ================================

if {"price_billion", "area_m2"}.issubset(df_viz.columns):
    plot_df = df_viz.dropna(subset=["price_billion", "area_m2"]).copy()

    hover_cols = [
        c for c in [
            "property_id",
            "title",
            "district",
            "location_zone",
            "property_type",
            "legal_class",
            "price_band",
            "area_band",
            "risk_level",
            "data_quality_level"
        ]
        if c in plot_df.columns
    ]

    color_col = "location_zone" if "location_zone" in plot_df.columns else None
    size_col = "price_per_m2_million" if "price_per_m2_million" in plot_df.columns else None

    fig = px.scatter(
        plot_df,
        x="area_m2",
        y="price_billion",
        color=color_col,
        size=size_col,
        hover_data=hover_cols,
        title="Quan hệ giữa diện tích và giá BĐS"
    )

    fig.update_layout(
        height=700,
        xaxis_title="Diện tích, m²",
        yaxis_title="Giá, tỷ VNĐ"
    )

    fig.show()

In [37]:
# ================================
# CELL 11 - SCATTER GIÁ/M² THEO QUẬN/HUYỆN
# ================================

if {"district", "price_per_m2_million"}.issubset(df_viz.columns):
    plot_df = df_viz.dropna(subset=["district", "price_per_m2_million"]).copy()

    fig = px.box(
        plot_df,
        x="district",
        y="price_per_m2_million",
        points="all",
        title="Phân bố giá/m² theo quận/huyện"
    )

    fig.update_layout(
        height=750,
        xaxis_title="Quận/Huyện",
        yaxis_title="Giá/m², triệu VNĐ",
        xaxis_tickangle=-45
    )

    fig.show()

In [38]:
# ================================
# CELL 12 - TRỰC QUAN AMENITY TAGS
# ================================

if "amenity_tags" in df_viz.columns:
    amenity_rows = []

    for _, row in df_viz.iterrows():
        tags = row["amenity_tags"]
        if not isinstance(tags, list):
            tags = []

        for tag in tags:
            amenity_rows.append({
                "property_id": row.get("property_id", None),
                "amenity_tag": tag
            })

    amenity_df = pd.DataFrame(amenity_rows)

    if len(amenity_df) > 0:
        amenity_count = (
            amenity_df["amenity_tag"]
            .value_counts()
            .reset_index()
        )
        amenity_count.columns = ["amenity_tag", "count"]

        fig = px.bar(
            amenity_count,
            x="count",
            y="amenity_tag",
            orientation="h",
            text="count",
            title="Tần suất tiện ích / đặc điểm ẩn được trích xuất"
        )
        fig.update_layout(height=650, xaxis_title="Số lượng", yaxis_title="Amenity tag")
        fig.show()

        display(amenity_count)
    else:
        print("Không có amenity_tags để trực quan hóa.")

,amenity_tag,count
0,business,221
1,rental,100
2,furnished,89
3,car_access,32
4,market,30
5,planning_infra,29
6,urgent_sale,28
7,school,20
8,park,17
9,security,15


In [39]:
# ================================
# CELL 13 - TRỰC QUAN SYMBOLIC FACTS
# ================================

if "symbolic_facts" in df_viz.columns:
    fact_rows = []

    for _, row in df_viz.iterrows():
        facts = row["symbolic_facts"]
        if not isinstance(facts, list):
            facts = []

        for fact in facts:
            fact_rows.append({
                "property_id": row.get("property_id", None),
                "symbolic_fact": fact
            })

    fact_df = pd.DataFrame(fact_rows)

    if len(fact_df) > 0:
        fact_count = (
            fact_df["symbolic_fact"]
            .value_counts()
            .reset_index()
        )
        fact_count.columns = ["symbolic_fact", "count"]

        fig = px.bar(
            fact_count.head(30),
            x="count",
            y="symbolic_fact",
            orientation="h",
            text="count",
            title="Top Symbolic Facts được suy luận"
        )
        fig.update_layout(height=850, xaxis_title="Số lượng", yaxis_title="Symbolic fact")
        fig.show()

        display(fact_count)
    else:
        print("Không có symbolic_facts để trực quan hóa.")

,symbolic_fact,count
0,low_legal_risk,208
1,suitable_for_family,114
2,suitable_for_investment,110
3,suitable_for_business,109
4,suitable_for_rental,95


In [40]:
# ================================
# CELL 14 - TRỰC QUAN RULES ĐƯỢC KÍCH HOẠT
# ================================

if "triggered_rules" in df_viz.columns:
    rule_rows = []

    for _, row in df_viz.iterrows():
        rules = row["triggered_rules"]
        if not isinstance(rules, list):
            rules = []

        for rule in rules:
            rule_rows.append({
                "property_id": row.get("property_id", None),
                "triggered_rule": rule
            })

    rule_df = pd.DataFrame(rule_rows)

    if len(rule_df) > 0:
        rule_count = (
            rule_df["triggered_rule"]
            .value_counts()
            .reset_index()
        )
        rule_count.columns = ["triggered_rule", "count"]

        fig = px.bar(
            rule_count,
            x="count",
            y="triggered_rule",
            orientation="h",
            text="count",
            title="Tần suất các luật Symbolic được kích hoạt"
        )
        fig.update_layout(height=650, xaxis_title="Số lần kích hoạt", yaxis_title="Rule ID")
        fig.show()

        display(rule_count)
    else:
        print("Không có triggered_rules để trực quan hóa.")

,triggered_rule,count
0,R01_LEGAL_CLEAR,208
1,R02_LEGAL_WARNING,122
2,R03_FAMILY_STRONG,114
3,R06_INVESTMENT_STRONG,110
4,R04_BUSINESS_STRONG,109
5,R05_RENTAL_STRONG,95
6,R08_OVERPRICED_WARNING,41
7,R07_DATA_LOW_CONFIDENCE,39


In [41]:
# ================================
# CELL 15 - TRỰC QUAN RISK FLAGS
# ================================

if "risk_flags" in df_viz.columns:
    risk_rows = []

    for _, row in df_viz.iterrows():
        risks = row["risk_flags"]
        if not isinstance(risks, list):
            risks = []

        for risk in risks:
            risk_rows.append({
                "property_id": row.get("property_id", None),
                "risk_flag": risk
            })

    risk_df = pd.DataFrame(risk_rows)

    if len(risk_df) > 0:
        risk_count = (
            risk_df["risk_flag"]
            .value_counts()
            .reset_index()
        )
        risk_count.columns = ["risk_flag", "count"]

        fig = px.bar(
            risk_count,
            x="count",
            y="risk_flag",
            orientation="h",
            text="count",
            title="Tần suất Risk Flags trong dữ liệu"
        )
        fig.update_layout(height=650, xaxis_title="Số lượng", yaxis_title="Risk flag")
        fig.show()

        display(risk_count)
    else:
        print("Không có risk_flags nào.")

,risk_flag,count
0,need_legal_verification,122
1,possibly_overpriced,41
2,low_data_confidence,39


In [42]:
# ================================
# CELL 16 - PHÂN BỐ CÁC ĐIỂM SYMBOLIC
# ================================

score_cols = [
    "legal_score",
    "price_reasonable_score",
    "family_suitability_score",
    "business_potential_score",
    "rental_potential_score",
    "investment_potential_score",
    "risk_score",
    "data_quality_score"
]

score_cols = [
    c for c in score_cols
    if c in df_viz.columns and pd.api.types.is_numeric_dtype(df_viz[c])
]

if len(score_cols) > 0:
    score_long = df_viz[score_cols].melt(
        var_name="score_type",
        value_name="score"
    )

    fig = px.box(
        score_long,
        x="score_type",
        y="score",
        title="So sánh phân bố các điểm Symbolic"
    )
    fig.update_layout(
        height=650,
        xaxis_title="Loại điểm Symbolic",
        yaxis_title="Điểm"
    )
    fig.show()

    fig = px.histogram(
        score_long,
        x="score",
        color="score_type",
        nbins=30,
        barmode="overlay",
        opacity=0.65,
        title="Phân bố các điểm Symbolic"
    )
    fig.update_layout(
        height=650,
        xaxis_title="Điểm",
        yaxis_title="Số lượng"
    )
    fig.show()
else:
    print("Không tìm thấy cột score phù hợp.")

In [43]:
# ================================
# CELL 17 - HEATMAP TƯƠNG QUAN GIỮA CÁC ĐIỂM SYMBOLIC
# ================================

if len(score_cols) >= 2:
    corr = df_viz[score_cols].corr()

    fig = px.imshow(
        corr,
        text_auto=True,
        title="Ma trận tương quan giữa các điểm Symbolic",
        aspect="auto"
    )

    fig.update_layout(height=700)
    fig.show()

    display(corr)
else:
    print("Cần ít nhất 2 cột score để vẽ heatmap.")

,legal_score,price_reasonable_score,family_suitability_score,business_potential_score,rental_potential_score,investment_potential_score,risk_score,data_quality_score
legal_score,1.000000,0.124450,0.794401,0.707437,0.743003,0.796105,-0.526165,0.654945
price_reasonable_score,0.124450,1.000000,0.197537,0.025952,0.153498,0.384851,-0.551093,0.272724
family_suitability_score,0.794401,0.197537,1.000000,0.706929,0.743021,0.747236,-0.589653,0.708147
business_potential_score,0.707437,0.025952,0.706929,1.000000,0.754347,0.721563,-0.471482,0.632295
rental_potential_score,0.743003,0.153498,0.743021,0.754347,1.000000,0.844724,-0.585571,0.700982
investment_potential_score,0.796105,0.384851,0.747236,0.721563,0.844724,1.000000,-0.693128,0.787728
risk_score,-0.526165,-0.551093,-0.589653,-0.471482,-0.585571,-0.693128,1.000000,-0.829285
data_quality_score,0.654945,0.272724,0.708147,0.632295,0.700982,0.787728,-0.829285,1.000000


In [44]:
# ================================
# CELL 18 - TOP BĐS THEO TỪNG ĐIỂM SYMBOLIC
# ================================

def show_top_properties_by_score(score_col, top_n=10):
    if score_col not in df_viz.columns:
        print(f"Không có cột {score_col}")
        return

    show_cols = [
        "property_id",
        "title",
        "district",
        "location_zone",
        "property_type",
        "price_billion",
        "area_m2",
        "price_per_m2_million",
        "legal_class",
        score_col,
        "risk_level",
        "data_quality_score"
    ]

    show_cols = [c for c in show_cols if c in df_viz.columns]

    top_df = (
        df_viz
        .sort_values(score_col, ascending=False)
        [show_cols]
        .head(top_n)
    )

    print(f"TOP {top_n} BĐS theo {score_col}")
    display(top_df)


for score_col in [
    "family_suitability_score",
    "business_potential_score",
    "rental_potential_score",
    "investment_potential_score",
    "price_reasonable_score",
    "data_quality_score"
]:
    if score_col in df_viz.columns:
        show_top_properties_by_score(score_col, top_n=10)

TOP 10 BĐS theo family_suitability_score


,property_id,title,district,location_zone,property_type,price_billion,area_m2,price_per_m2_million,legal_class,family_suitability_score,risk_level,data_quality_score
58,BDS_00059,"Mặt tiền lớn đường số gần bệnh viện Lê Văn Thịnh. Ở hoặc mở VP công ty đều tiện, xe hơi đậu sân",Quận 2,developing,land,9.90,74.00,133.783784,clear_ownership,1.00,very_low,1.0
46,BDS_00047,"Mặt tiền lô H Cư Xá Phú Lâm B Quận 6 | DT 4 x 18m, trệt + lầu, giá chỉ 8 tỷ",Quận 6,urban,townhouse,8.00,66.30,120.663650,clear_ownership,0.95,very_low,1.0
143,BDS_00144,"Nhà phố 4m x 19m, 4 lầu 6PN, 7WC, mặt tiền đường Tân Thới Hiệp 9 (tòa án cũ)",Quận 12,suburban,townhouse,8.79,76.00,115.657895,clear_ownership,0.95,very_low,1.0
47,BDS_00048,"MT đường 1A khu Tên Lửa, 4x21m, 2 tầng, chỉ 9.8 tỷ",Bình Tân,suburban,townhouse,9.90,84.00,117.857143,clear_ownership,0.93,very_low,1.0
28,BDS_00029,"Bán tòa nhà ngang 14.15m mặt tiền Đặng Dung Q1, 14.15x24.17m - Hầm 7 tầng - HDT 280tr - Giá 182tỷ",Quận 1,central,townhouse,182.00,339.59,535.940399,clear_ownership,0.90,very_low,1.0
24,BDS_00025,"Mình chính chủ cần bán đất có nhà tại Trung Chánh - Bà Điểm, HM (gần ngã 4 Tô Ký - Nguyễn Ảnh Thủ)",Hóc Môn,suburban,land,8.80,217.00,40.552995,clear_ownership,0.90,very_low,1.0
3,BDS_00004,"Bán nhà mặt tiền Nguyễn Thị Sóc, Bà Điểm, TPHCM. 168m2 (6x27m) đang thuê 25tr/th sổ hồng riêng",Hóc Môn,suburban,townhouse,16.50,168.00,98.214286,clear_ownership,0.90,very_low,1.0
5,BDS_00006,Nhà mặt tiền KDC Bình Hưng sát Pegasus Tạ Quang Bửu - 96m2 - 4 tầng chỉ 9.9tỷ,Bình Chánh,developing,townhouse,9.90,96.00,103.125000,clear_ownership,0.90,very_low,1.0
117,BDS_00118,Mặt tiền kinh doanh - nhà 4 tầng - 87.18m² (5.3m x 16.5m) - 4PN - Full nội thất,Tân Phú,urban,townhouse,15.95,87.18,182.954806,clear_ownership,0.89,very_low,1.0
119,BDS_00120,"Siêu phẩm! Góc 2 mặt tiền đường Phạm Văn Hai - gần chợ, 4,5mx18m, 5 tầng, HĐT 45tr, chỉ 23 tỷ",Tân Bình,urban,townhouse,23.00,85.00,270.588235,clear_ownership,0.89,very_low,1.0


TOP 10 BĐS theo business_potential_score


,property_id,title,district,location_zone,property_type,price_billion,area_m2,price_per_m2_million,legal_class,business_potential_score,risk_level,data_quality_score
176,BDS_00177,"Bán nhà mặt tiền đường Nguyễn Trường Tộ, P. Tân Thành - DT: 4x18m vuông vức - Giá tốt: 10,5 tỷ",Tân Phú,urban,townhouse,10.5,72.00,145.833333,clear_ownership,0.94,very_low,1.0
40,BDS_00041,"Bán nhà MTKD Khu Phạm Viết Chánh sát Bến Nghé, Quận 1. DT(4mx10m) 4 tầng. Sẵn HĐT 30 triệu/tháng",Bình Thạnh,central,townhouse,10.5,37.60,279.255319,clear_ownership,0.94,very_low,1.0
92,BDS_00093,"Toà nhà giảm giá 50,9tỷ giảm về 47tỷ, ngay BV Nhi Đồng 1, NH(6x20m) hầm trệt 5 lầu ST, vỉa hè 10m",Quận 10,central,townhouse,47.0,125.00,376.000000,clear_ownership,0.94,very_low,1.0
28,BDS_00029,"Bán tòa nhà ngang 14.15m mặt tiền Đặng Dung Q1, 14.15x24.17m - Hầm 7 tầng - HDT 280tr - Giá 182tỷ",Quận 1,central,townhouse,182.0,339.59,535.940399,clear_ownership,0.90,very_low,1.0
81,BDS_00082,Khách sạn mặt tiền Phố Tây Bùi Viện - Cách Chợ Bến Thành 300m - 8x20 - Hầm 9 tầng - 110 Tỷ,Quận 1,central,townhouse,110.0,153.30,717.547293,clear_ownership,0.90,very_low,1.0
148,BDS_00149,"Building 10 tầng mặt tiền Âu Cơ, Q. Tân Bình - có sẵn HĐT 300 triệu/tháng. Giá 100 tỷ về bán 85 tỷ",Tân Bình,urban,townhouse,85.0,238.00,357.142857,clear_ownership,0.90,very_low,1.0
147,BDS_00148,"Mặt tiền Lũy Bán Bích 8.6x25m đúc 3.5 tấm (35 tỷ). Vị trí đắc địa, sầm uất, sẵn HĐ thuê 80 tr/th",Tân Phú,urban,townhouse,35.0,127.00,275.590551,clear_ownership,0.90,very_low,1.0
94,BDS_00095,"Toà nhà góc 2 mặt tiền Nguyễn Sơn, Q. Tân Phú, 10x28, hầm 6 tầng, HĐT 238 triệu/ chỉ 52.8 tỷ",Tân Phú,urban,townhouse,52.8,172.00,306.976744,clear_ownership,0.90,very_low,1.0
35,BDS_00036,Building 6 tầng góc 2MT quận 1 cạnh Công viên Lê Văn Tám Vị trí giá giảm cục sâu chỉ 64 tỷ,Quận 1,central,townhouse,64.0,105.00,609.523810,other,0.90,very_low,1.0
48,BDS_00049,"Bán nhà mặt tiền đường Thất Sơn, khu Cư Xá Bắc Hải, quận 10. DT 6x25m hầm 4 tầng 36,5 tỷ TL",Quận 10,central,townhouse,36.5,152.00,240.131579,clear_ownership,0.90,very_low,1.0


TOP 10 BĐS theo rental_potential_score


...

,property_id,title,district,location_zone,property_type,price_billion,area_m2,price_per_m2_million,legal_class,investment_potential_score,risk_level,data_quality_score
79,BDS_00080,Siêu phẩm 2 mặt tiền Nguyễn Kiệm hiếm có 4.46x24m giá tốt chỉ 166tr/m2,Phú Nhuận,central,townhouse,19.00,113.8,166.959578,clear_ownership,0.940,very_low,1.0
18,BDS_00019,"Khu sang giá tốt! MT - Hoàng Diệu, P10, PN 4T chỉ 19 tỷ SH giao ngay LH: Mr Trung0971 291 ***",Phú Nhuận,central,apartment,19.00,65.4,290.519878,clear_ownership,0.918,very_low,1.0
64,BDS_00065,"Hiếm có, mặt tiền Hoàng Diệu, sát Nguyễn Văn Trỗi, 4x16,5m, 5 tầng kiên cố, ở ngay, chỉ 19 tỷ TL",Phú Nhuận,central,townhouse,19.00,66.0,287.878788,clear_ownership,0.918,very_low,1.0
185,BDS_00186,"TP Thủ Đức - MT Nguyễn Trung Nguyệt - 70.9m² - 8.35 tỷ - Gần Quận 1, Bình Thạnh",Quận 2,developing,townhouse,8.35,70.9,117.771509,clear_ownership,0.910,very_low,1.0
189,BDS_00190,"Rẻ hơn thị trường 6 tỷ! Bán gấp tòa nhà 4 tầng MT Bạch Đằng, TB, diện tích: 6.2x20m, giá: 23 tỷ",Tân Bình,urban,townhouse,23.00,120.0,191.666667,clear_ownership,0.910,very_low,1.0
78,BDS_00079,"Bán nhà C4 mặt tiền Thạnh Mỹ Lợi chỉ 98tr/m2 (5 x 27,6)",Quận 2,developing,land,13.50,137.7,98.039216,clear_ownership,0.910,very_low,1.0
77,BDS_00078,"Nhà mặt tiền Võ Văn Hát 5m x 21,5m, dòng tiền cho thuê sẵn, Giá 8tỷ",Quận 9,developing,townhouse,8.00,107.0,74.766355,clear_ownership,0.910,very_low,1.0
45,BDS_00046,"4.3 tỷ | 5 tầng - 4 PN | MT Đường Số Lý Phục Man, vỉa hè 1.5m | Sơn lại nhẹ cho thuê 15tr/th",Quận 7,developing,townhouse,4.30,17.4,247.126437,clear_ownership,0.888,very_low,1.0
5,BDS_00006,Nhà mặt tiền KDC Bình Hưng sát Pegasus Tạ Quang Bửu - 96m2 - 4 tầng chỉ 9.9tỷ,Bình Chánh,developing,townhouse,9.90,96.0,103.125000,clear_ownership,0.888,very_low,1.0
201,BDS_00202,"Hạ giá bán gấp 3PN Vinhomes Central Park, khu Park view sông",Bình Thạnh,central,apartment,13.90,118.0,117.796610,clear_ownership,0.860,very_low,1.0


TOP 10 BĐS theo price_reasonable_score


,property_id,title,district,location_zone,property_type,price_billion,area_m2,price_per_m2_million,legal_class,price_reasonable_score,risk_level,data_quality_score
329,BDS_00330,,Không rõ,unknown,unknown,5.0,80.00,62.500000,other,1.0,very_low,0.75
0,BDS_00001,"Không mua căn này thì mua căn nào ạ! 5,7x13,3m 3 tầng KDC Him Lam - ở + kinh doanh tuyệt vời",Thủ Đức,developing,townhouse,8.9,76.00,117.105263,clear_ownership,1.0,very_low,1.00
325,BDS_00326,,Không rõ,unknown,house,5.0,67.20,74.404762,other,1.0,very_low,0.80
299,BDS_00300,,Không rõ,unknown,townhouse,4.0,105.00,38.095238,other,1.0,very_low,0.80
309,BDS_00310,,Không rõ,unknown,house,3.0,38.64,77.639752,clear_ownership,1.0,very_low,0.80
301,BDS_00302,,Không rõ,unknown,house,4.0,49.50,80.808081,other,1.0,very_low,0.80
304,BDS_00305,,Không rõ,unknown,house,5.0,75.60,66.137566,other,1.0,very_low,0.80
36,BDS_00037,Sở hữu thế đất viên kim cương Panorama 360 độ - nhà 6 tầng mặt tiền Hoàng Sa. Giá tốt 13.5 tỷ,Quận 1,central,land,13.5,36.16,373.340708,clear_ownership,1.0,very_low,1.00
32,BDS_00033,"Dòng tiền 220 triệu/tháng! Toà building mặt tiền Hồ Bá Kiện, Quận 10 (8,5x20m, hầm 6 tầng) - 42 tỷ",Quận 10,central,townhouse,42.0,174.50,240.687679,clear_ownership,1.0,very_low,1.00
296,BDS_00297,,Không rõ,unknown,house,4.0,49.00,81.632653,other,1.0,very_low,0.80


TOP 10 BĐS theo data_quality_score


,property_id,title,district,location_zone,property_type,price_billion,area_m2,price_per_m2_million,legal_class,data_quality_score,risk_level,data_quality_score
327,BDS_00328,,Gò Vấp,urban,house,6.50,60.0,108.333333,other,1.0,very_low,1.0
0,BDS_00001,"Không mua căn này thì mua căn nào ạ! 5,7x13,3m 3 tầng KDC Him Lam - ở + kinh doanh tuyệt vời",Thủ Đức,developing,townhouse,8.90,76.0,117.105263,clear_ownership,1.0,very_low,1.0
1,BDS_00002,"Nhà MT Lạc Long Quân-Q11. Ngang 6m x 15m, sổ vuông (nở hậu 6,8m), 3 tầng, CN: 95m2, vị trí đắc địa",Quận 11,urban,townhouse,30.00,95.0,315.789474,clear_ownership,1.0,very_low,1.0
324,BDS_00325,,Gò Vấp,urban,townhouse,4.00,33.3,120.120120,clear_ownership,1.0,very_low,1.0
3,BDS_00004,"Bán nhà mặt tiền Nguyễn Thị Sóc, Bà Điểm, TPHCM. 168m2 (6x27m) đang thuê 25tr/th sổ hồng riêng",Hóc Môn,suburban,townhouse,16.50,168.0,98.214286,clear_ownership,1.0,very_low,1.0
288,BDS_00289,,Quận 12,suburban,apartment,3.00,58.8,51.020408,other,1.0,very_low,1.0
287,BDS_00288,,Quận 12,suburban,apartment,3.00,58.8,51.020408,other,1.0,very_low,1.0
284,BDS_00285,,Gò Vấp,urban,townhouse,2.00,27.0,74.074074,other,1.0,very_low,1.0
283,BDS_00284,,Gò Vấp,urban,house,4.90,57.0,85.964912,other,1.0,very_low,1.0
280,BDS_00281,,Gò Vấp,urban,townhouse,3.25,30.0,108.333333,other,1.0,very_low,1.0


In [45]:
# ================================
# CELL 19 - TẠO OVERALL SYMBOLIC SCORE VÀ BẢNG XẾP HẠNG
# ================================

overall_components = [
    "legal_score",
    "price_reasonable_score",
    "family_suitability_score",
    "business_potential_score",
    "rental_potential_score",
    "investment_potential_score",
    "data_quality_score"
]

overall_components = [
    c for c in overall_components
    if c in df_viz.columns and pd.api.types.is_numeric_dtype(df_viz[c])
]

df_rank = df_viz.copy()

if len(overall_components) > 0:
    df_rank["symbolic_overall_score"] = df_rank[overall_components].mean(axis=1, skipna=True)

    if "risk_score" in df_rank.columns:
        df_rank["symbolic_overall_score_adjusted"] = (
            df_rank["symbolic_overall_score"] * (1 - 0.25 * df_rank["risk_score"].fillna(0))
        )
    else:
        df_rank["symbolic_overall_score_adjusted"] = df_rank["symbolic_overall_score"]

    show_cols = [
        "property_id",
        "title",
        "district",
        "location_zone",
        "property_type",
        "price_billion",
        "area_m2",
        "price_per_m2_million",
        "legal_class",
        "family_suitability_score",
        "business_potential_score",
        "rental_potential_score",
        "investment_potential_score",
        "risk_score",
        "data_quality_score",
        "symbolic_overall_score",
        "symbolic_overall_score_adjusted",
        "risk_flags"
    ]

    show_cols = [c for c in show_cols if c in df_rank.columns]

    top_symbolic_df = (
        df_rank
        .sort_values("symbolic_overall_score_adjusted", ascending=False)
        [show_cols]
        .head(30)
    )

    display(top_symbolic_df)

    fig = px.histogram(
        df_rank,
        x="symbolic_overall_score_adjusted",
        nbins=30,
        title="Phân bố điểm Symbolic Overall đã hiệu chỉnh theo rủi ro"
    )
    fig.update_layout(
        xaxis_title="Symbolic overall score adjusted",
        yaxis_title="Số lượng"
    )
    fig.show()
else:
    print("Không đủ cột score để tạo overall score.")

,property_id,title,district,location_zone,property_type,price_billion,area_m2,price_per_m2_million,legal_class,family_suitability_score,business_potential_score,rental_potential_score,investment_potential_score,risk_score,data_quality_score,symbolic_overall_score,symbolic_overall_score_adjusted,risk_flags
164,BDS_00165,Ngộp bank bán gấp MT Nguyễn Thiện Thuật Q3-giảm sốc 8 tỷ-93m2-4 tầng chỉ còn 33 tỷ-LH0789 162 ***,Quận 3,central,townhouse,33.0,93.00,354.838710,clear_ownership,0.85,0.84,0.93,0.860,0.0,1.0,0.925714,0.925714,[]
92,BDS_00093,"Toà nhà giảm giá 50,9tỷ giảm về 47tỷ, ngay BV Nhi Đồng 1, NH(6x20m) hầm trệt 5 lầu ST, vỉa hè 10m",Quận 10,central,townhouse,47.0,125.00,376.000000,clear_ownership,0.85,0.94,0.93,0.838,0.0,1.0,0.922571,0.922571,[]
129,BDS_00130,"Góc 2 mặt tiền đường Lê Văn Sỹ 5.4x18m, 5 tầng HĐT 100tr/th chủ thiện chí bán gấp 38 tỷ TL",Quận 3,central,townhouse,38.0,98.20,386.965377,clear_ownership,0.85,0.84,0.87,0.860,0.0,1.0,0.917143,0.917143,[]
6,BDS_00007,"Góc 2 mặt tiền Trần Đình Xu, 4.1x18m, nhà 3 lầu HĐ thuê 60tr/th, giá 24.8 tỷ TL",Quận 1,central,townhouse,24.8,72.80,340.659341,clear_ownership,0.85,0.84,0.87,0.800,0.0,1.0,0.908571,0.908571,[]
96,BDS_00097,"Nhà mặt tiền kinh doanh Huỳnh Tấn Phát TT Nhà Bè - 6x30m - xây khách sạn - CHDV - giá 19,8 tỷ",Nhà Bè,developing,townhouse,19.8,182.90,108.255878,clear_ownership,0.85,0.82,0.91,0.770,0.0,1.0,0.907143,0.907143,[]
160,BDS_00161,"Ngộp bank! Mặt tiền Nguyễn Xí DT: 4.2x12m KC 1 hầm 6 lầu giá 13.5tỷ, HDT: 65 tr/th, gần DH Văn Lang",Bình Thạnh,central,townhouse,13.5,52.20,258.620690,clear_ownership,0.88,0.84,0.87,0.838,0.0,1.0,0.904000,0.904000,[]
5,BDS_00006,Nhà mặt tiền KDC Bình Hưng sát Pegasus Tạ Quang Bửu - 96m2 - 4 tầng chỉ 9.9tỷ,Bình Chánh,developing,townhouse,9.9,96.00,103.125000,clear_ownership,0.90,0.76,0.81,0.888,0.0,1.0,0.894000,0.894000,[]
128,BDS_00129,"Bán nhà mặt tiền Trương Định, Quận 3 vị trí đẹp, nhà 6 lầu hiện đại, giá 28.6 tỷ TL",Quận 3,central,townhouse,28.6,64.80,441.358025,clear_ownership,0.80,0.84,0.87,0.838,0.0,1.0,0.892571,0.892571,[]
168,BDS_00169,"Mặt tiền đường Lê Hồng Phong 4.7x22m, 5 tầng gần Nguyễn Trãi - An Dương Vương giá 35 tỷ TL",Quận 5,central,townhouse,35.0,106.30,329.256820,clear_ownership,0.85,0.84,0.87,0.778,0.0,1.0,0.891143,0.891143,[]
176,BDS_00177,"Bán nhà mặt tiền đường Nguyễn Trường Tộ, P. Tân Thành - DT: 4x18m vuông vức - Giá tốt: 10,5 tỷ",Tân Phú,urban,townhouse,10.5,72.00,145.833333,clear_ownership,0.80,0.94,0.91,0.688,0.0,1.0,0.891143,0.891143,[]


In [46]:
# ================================
# CELL 20 - RADAR CHART SO SÁNH 2-3 BĐS
# ================================

def radar_compare_properties(property_ids):
    if "property_id" not in df_viz.columns:
        raise ValueError("Không có cột property_id.")

    radar_cols = [
        "legal_score",
        "price_reasonable_score",
        "family_suitability_score",
        "business_potential_score",
        "rental_potential_score",
        "investment_potential_score",
        "data_quality_score"
    ]

    radar_cols = [
        c for c in radar_cols
        if c in df_viz.columns and pd.api.types.is_numeric_dtype(df_viz[c])
    ]

    if len(radar_cols) == 0:
        raise ValueError("Không có cột score để vẽ radar.")

    sub = df_viz[df_viz["property_id"].isin(property_ids)].copy()

    if len(sub) == 0:
        raise ValueError("Không tìm thấy property_id đã nhập.")

    fig = go.Figure()

    for _, row in sub.iterrows():
        values = [row[c] if not pd.isna(row[c]) else 0 for c in radar_cols]
        values = values + [values[0]]
        categories = radar_cols + [radar_cols[0]]

        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=categories,
            fill="toself",
            name=str(row["property_id"])
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="Radar chart so sánh điểm Symbolic giữa các BĐS",
        height=700
    )

    fig.show()


sample_ids = df_viz["property_id"].dropna().head(3).tolist() if "property_id" in df_viz.columns else []

print("Sample property_ids:", sample_ids)

if len(sample_ids) >= 2:
    radar_compare_properties(sample_ids)

Sample property_ids: ['BDS_00001', 'BDS_00002', 'BDS_00003']


In [47]:
# ================================
# CELL 21 - BIỂU ĐỒ SUNBURST: ZONE -> DISTRICT -> PROPERTY TYPE
# ================================

required_cols = {"location_zone", "district", "property_type"}

if required_cols.issubset(df_viz.columns):
    sunburst_df = (
        df_viz
        .fillna({
            "location_zone": "unknown",
            "district": "Không rõ",
            "property_type": "unknown"
        })
        .groupby(["location_zone", "district", "property_type"])
        .size()
        .reset_index(name="count")
    )

    fig = px.sunburst(
        sunburst_df,
        path=["location_zone", "district", "property_type"],
        values="count",
        title="Cấu trúc dữ liệu theo nhóm vị trí → quận/huyện → loại BĐS"
    )

    fig.update_layout(height=750)
    fig.show()

    display(sunburst_df)
else:
    print("Thiếu cột location_zone, district hoặc property_type.")

,location_zone,district,property_type,count
0,central,Bình Thạnh,apartment,1
1,central,Bình Thạnh,land,2
2,central,Bình Thạnh,townhouse,5
3,central,Phú Nhuận,apartment,1
4,central,Phú Nhuận,land,7
...,...,...,...,...
56,urban,Tân Bình,townhouse,14
57,urban,Tân Bình,unknown,1
58,urban,Tân Bình,villa,2
59,urban,Tân Phú,land,4


In [48]:
# ================================
# CELL 22 - BIỂU ĐỒ TREEMAP: PHÁP LÝ -> RỦI RO -> LOẠI BĐS
# ================================

required_cols = {"legal_class", "risk_level", "property_type"}

if required_cols.issubset(df_viz.columns):
    tree_df = (
        df_viz
        .fillna({
            "legal_class": "unknown",
            "risk_level": "unknown",
            "property_type": "unknown"
        })
        .groupby(["legal_class", "risk_level", "property_type"])
        .size()
        .reset_index(name="count")
    )

    fig = px.treemap(
        tree_df,
        path=["legal_class", "risk_level", "property_type"],
        values="count",
        title="Treemap pháp lý → mức rủi ro → loại BĐS"
    )

    fig.update_layout(height=750)
    fig.show()

    display(tree_df)
else:
    print("Thiếu cột legal_class, risk_level hoặc property_type.")


,legal_class,risk_level,property_type,count
0,clear_ownership,low,house,3
1,clear_ownership,low,townhouse,3
2,clear_ownership,low,unknown,3
3,clear_ownership,very_low,apartment,3
4,clear_ownership,very_low,house,7
5,clear_ownership,very_low,land,35
6,clear_ownership,very_low,townhouse,146
7,clear_ownership,very_low,unknown,5
8,clear_ownership,very_low,villa,3
9,contract_based,very_low,apartment,1


In [49]:
# ================================
# CELL 23 - LƯU CÁC BẢNG THỐNG KÊ RA EXCEL
# ================================

OUTPUT_VIZ_DIR = Path("/content/bds_symbolic_visualization")
OUTPUT_VIZ_DIR.mkdir(exist_ok=True)

summary_path = OUTPUT_VIZ_DIR / "bds_symbolic_visualization_summary.xlsx"

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
    df_viz.head(100).to_excel(writer, sheet_name="sample_data", index=False)

    if "district" in df_viz.columns:
        df_viz["district"].fillna("Không rõ").value_counts().reset_index().to_excel(
            writer,
            sheet_name="district_count",
            index=False
        )

    if "location_zone" in df_viz.columns:
        df_viz["location_zone"].fillna("unknown").value_counts().reset_index().to_excel(
            writer,
            sheet_name="zone_count",
            index=False
        )

    if "legal_class" in df_viz.columns:
        df_viz["legal_class"].fillna("unknown").value_counts().reset_index().to_excel(
            writer,
            sheet_name="legal_count",
            index=False
        )

    if "property_type" in df_viz.columns:
        df_viz["property_type"].fillna("unknown").value_counts().reset_index().to_excel(
            writer,
            sheet_name="property_type_count",
            index=False
        )

    if "amenity_count" in globals():
        amenity_count.to_excel(writer, sheet_name="amenity_count", index=False)

    if "fact_count" in globals():
        fact_count.to_excel(writer, sheet_name="symbolic_fact_count", index=False)

    if "rule_count" in globals():
        rule_count.to_excel(writer, sheet_name="rule_count", index=False)

    if "risk_count" in globals():
        risk_count.to_excel(writer, sheet_name="risk_count", index=False)

    if "top_symbolic_df" in globals():
        top_symbolic_df.to_excel(writer, sheet_name="top_symbolic", index=False)

    if "district_price" in globals():
        district_price.to_excel(writer, sheet_name="district_price", index=False)

    if "district_ppm" in globals():
        district_ppm.to_excel(writer, sheet_name="district_price_m2", index=False)

print("Đã lưu file thống kê tại:")
print(summary_path)

Đã lưu file thống kê tại:
/content/bds_symbolic_visualization/bds_symbolic_visualization_summary.xlsx
